# Terra Signal | Análise, Base Analítica e Modelagem de Churn
**Hackathon Databricks 2026 · Grupo 6**

## 00. Objetivo

A Terra Signal é uma empresa de telecomunicações. O objetivo é desenvolver um modelo que identifique clientes com maior **risco de cancelamento de contrato (churn)**.

O desafio pede uma classificação **Yes/No** para cada cliente da base de inferência. O modelo também gera uma probabilidade estimada, que permite organizar os clientes em um ranking de risco.

O notebook reúne conferência dos arquivos, dicionário de dados, análise exploratória e preparação da base analítica (ABT) (seções 01–08), seguidas da modelagem em seis passos cronológicos (seções 09–18).

**Limitação:** os arquivos não possuem datas para acompanhar os cancelamentos ao longo do tempo. Assim, a avaliação feita aqui mostra o desempenho nos dados históricos disponíveis, mas não comprova com que antecedência um cancelamento poderia ser previsto.

### Dependências

O XGBoost é instalado no ambiente do notebook para participar da comparação de algoritmos.

In [0]:
%pip install xgboost

In [0]:
# Reinicia o Python após instalar bibliotecas, antes de carregar dados e treinar modelos.
dbutils.library.restartPython()

## 01. Bases recebidas

Foram utilizados dois arquivos: `history.csv`, com o resultado conhecido de `Churn` (**cancelamento**), e `inference.csv`, que reúne os clientes para os quais será feita a previsão.

### 01.1. Leitura e estrutura dos arquivos

In [0]:
# 01.1 | Carregar e conferir os arquivos
import pandas as pd
from pathlib import Path

# Localiza os dois CSVs na pasta do projeto, sem depender do usuário ou da máquina.
def localizar_pasta_dados():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    try:
        # Em notebooks Databricks, o diretório de execução pode ser diferente da pasta do notebook.
        caminho = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        base = Path(caminho)
        candidatos = [Path('/Workspace') / caminho.lstrip('/'), base, base.parent, *candidatos]
    except Exception:
        pass  # Fora do Databricks, usa a pasta de execução do notebook.

    for candidato in candidatos:
        pasta = candidato if candidato.is_dir() else candidato.parent
        if (pasta / 'history.csv').is_file() and (pasta / 'inference.csv').is_file():
            return pasta
    raise FileNotFoundError('history.csv e inference.csv devem estar juntos na pasta do projeto/notebook.')

pasta_dados = localizar_pasta_dados()
df_historico = pd.read_csv(pasta_dados / 'history.csv')
df_inferencia = pd.read_csv(pasta_dados / 'inference.csv')

resumo_bases = pd.DataFrame([
    {"Base": "Histórico", "Clientes": len(df_historico), "Variáveis": df_historico.shape[1]},
    {"Base": "Inferência", "Clientes": len(df_inferencia), "Variáveis": df_inferencia.shape[1]}
])
display(resumo_bases)

print("Colunas presentes apenas no histórico:",
      sorted(set(df_historico.columns) - set(df_inferencia.columns)))
print("Colunas de data/timestamp:",
      [c for c in df_historico.columns if any(
          chave in c.lower() for chave in ["date", "timestamp", "_data", "data_"]
      )])


### 01.2. Resultado da leitura

A base histórica contém **5.634 clientes e 23 colunas**; a de inferência contém **1.409 clientes e 22 colunas**. A diferença é a coluna `Churn` (**cancelamento**), disponível apenas no histórico. Não há colunas de data nos arquivos.

## 02. Conferência da quantidade de registros

A apresentação do desafio cita 5.912 linhas no histórico e 1.461 na inferência, mas os arquivos contêm menos registros de clientes. A verificação abaixo diferencia **linhas físicas do arquivo** de **registros válidos do CSV**.

Isso é importante porque `CustomerFeedback` (**relato do cliente**) pode conter quebras de linha dentro de um mesmo texto.

### 02.1. Validação dos registros

In [0]:
# 02.1 | Conferir linhas físicas e registros CSV
import csv

# Arquivos de entrada
arquivos = [pasta_dados / "history.csv", pasta_dados / "inference.csv"]

for arquivo in arquivos:

    with open(arquivo, "r", encoding="utf-8", newline="") as f:
        leitor = csv.reader(f)

        cabecalho = next(leitor)
        registros = [linha for linha in leitor if linha]

        linhas_fisicas = leitor.line_num - 1

        registros_invalidos = sum(
            len(linha) != len(cabecalho)
            for linha in registros
        )

    print(f"\nArquivo: {arquivo.name}")
    print(f"Linhas físicas (sem cabeçalho): {linhas_fisicas}")
    print(f"Registros interpretados: {len(registros)}")
    print(f"Quantidade de colunas: {len(cabecalho)}")
    print(f"Registros com número incorreto de campos: {registros_invalidos}")


### 02.2. Resultado da conferência

| Base | Linhas físicas | Registros de clientes | Registros com quantidade incorreta de campos |
|---|---:|---:|---:|
| Histórico | 5.912 | 5.634 | 0 |
| Inferência | 1.461 | 1.409 | 0 |

**Conclusão:** a diferença ocorre porque um mesmo relato pode ocupar várias linhas do arquivo. Não foram encontrados registros com número incorreto de campos. Para as análises, foi considerado **um registro por cliente**, e não cada linha física.

## 03. Dicionário de dados — arquivos originais

A tabela apresenta as **23 variáveis do histórico**, seus significados, valores observados, tipos, percentuais de ausência e decisões de tratamento. Os indicadores são calculados nos arquivos originais, antes das transformações.

### 03.1. Dicionário das variáveis

In [0]:
# 03.1 | Montar o dicionário dos dados originais
metadados = {
    "customerID": ("ID do cliente", "Identificador único do cliente", "Identificador único do cliente", "Identificação; excluir das preditoras"),
    "gender": ("Gênero", "Gênero informado do cliente", "Male = masculino; Female = feminino", "Sem tratamento específico definido"),
    "SeniorCitizen": ("Cliente idoso", "Indicador de cliente com 65 anos ou mais", "0 = não idoso; 1 = idoso", "Tratar como categórica binária"),
    "Partner": ("Companheiro(a)", "Indica se possui companheiro(a)", "Yes = possui companheiro(a); No = não possui", "Sem tratamento específico definido"),
    "Dependents": ("Dependentes", "Indica se possui dependentes", "Yes = possui; No = não possui", "Sem tratamento específico definido"),
    "tenure": ("Tempo de relacionamento", "Tempo em meses até a referência do registro", "0 a 72 meses; unknown = valor não informado", "Converter para numérico; estimar por TotalCharges / MonthlyCharges quando possível; referência temporal incerta"),
    "PhoneService": ("Serviço telefônico", "Possui serviço de telefonia", "Yes/yes = possui telefone; No/no = não possui", "Padronizar Yes/No"),
    "MultipleLines": ("Múltiplas linhas", "Possui múltiplas linhas telefônicas", "Yes = possui; No = não possui; No phone service = sem telefone", "Preservar 'No phone service' como categoria válida"),
    "InternetService": ("Serviço de internet", "Tipo de serviço de internet", "Fiber optic = fibra; DSL = DSL; No = sem internet", "Preservar categorias de serviço"),
    "OnlineSecurity": ("Segurança online", "Possui serviço de segurança online", "Yes = possui; No = não possui; No internet service = não aplicável", "Preservar não aplicável"),
    "OnlineBackup": ("Backup online", "Possui serviço de backup online", "Yes = possui; No = não possui; No internet service = não aplicável", "Preservar não aplicável"),
    "DeviceProtection": ("Proteção de dispositivos", "Possui proteção de dispositivos", "Yes = possui; No = não possui; No internet service = não aplicável", "Preservar não aplicável"),
    "TechSupport": ("Suporte técnico", "Possui serviço de suporte técnico", "Yes = possui; No = não possui; No internet service = não aplicável", "Preservar não aplicável"),
    "StreamingTV": ("Streaming de TV", "Possui serviço de streaming de TV", "Yes = possui; No = não possui; No internet service = não aplicável", "Preservar não aplicável"),
    "StreamingMovies": ("Streaming de filmes", "Possui serviço de streaming de filmes", "Yes = possui; No = não possui; No internet service = não aplicável", "Preservar não aplicável"),
    "Contract": ("Contrato", "Modalidade contratual vigente", "Month-to-month = mensal; One year = anual; Two year = bienal", "Não inferir histórico de renovações ou meses restantes"),
    "PaperlessBilling": ("Fatura digital", "Utiliza faturamento sem papel", "Yes = fatura digital; No = não", "Sem tratamento específico definido"),
    "PaymentMethod": ("Forma de pagamento", "Modalidade de pagamento", "Electronic check; Mailed check; Credit card (automatic); Bank transfer (automatic)", "Sem tratamento específico definido"),
    "MonthlyCharges": ("Cobrança mensal", "Valor da última mensalidade faturada", "18.4 a 118.65", "Pode variar ao longo do contrato; verificar relações com tenure e TotalCharges"),
    "TotalCharges": ("Cobrança acumulada", "Montante acumulado efetivamente pago desde a contratação", "18.8 a 8684.8", "Estimar com tenure × MonthlyCharges quando ausente; aproximação, não valor observado"),
    "Churn": ("Cancelamento", "Indica se o cliente cancelou", "Yes = cancelou (1); No = não cancelou (0)", "Resposta (Y); Yes = classe positiva"),
    "CustomerFeedback": ("Feedback do cliente", "Relato textual do cliente", "Texto livre fornecido pelo cliente", "Excluir do modelo principal: risco de leakage temporal"),
    "MonthlyIncome": ("Renda mensal", "Renda mensal familiar informada", "100 a 9993", "Não confundir com renda individual")
}

def contar_ausentes(serie):
    mascara = serie.isna()
    if pd.api.types.is_object_dtype(serie) or pd.api.types.is_string_dtype(serie):
        mascara = mascara | serie.astype('string').str.strip().str.lower().isin(['', 'unknown', 'null']).fillna(False)
    return int(mascara.sum())

registros = []
for coluna in df_historico.columns:
    traducao, significado, codificacao, cuidado = metadados[coluna]
    h = df_historico[coluna]; miss_h = contar_ausentes(h)
    pct_i = 100 * contar_ausentes(df_inferencia[coluna]) / len(df_inferencia) if coluna in df_inferencia else None
    papel = ('Identificador' if coluna == 'customerID' else 'Resposta (Y)' if coluna == 'Churn'
             else 'Excluída do modelo inicial — risco de leakage' if coluna == 'CustomerFeedback' else 'Preditora candidata (X)')
    registros.append({
        'Variável original': coluna, 'Tradução': traducao, 'Significado de negócio': significado,
        'Valores observados / codificação': codificacao, 'Papel': papel, 'Tipo Pandas': str(h.dtype),
        'Missing histórico': miss_h, '% Missing histórico': round(100 * miss_h / len(df_historico), 2),
        '% Missing inferência': round(pct_i, 2) if pct_i is not None else None,
        'Valores distintos': int(h.nunique(dropna=True)), 'Atenção / decisão': cuidado
    })

colunas_dicionario = ['Variável original', 'Tradução', 'Significado de negócio', 'Valores observados / codificação',
                     'Papel', 'Tipo Pandas', 'Missing histórico', '% Missing histórico', '% Missing inferência',
                     'Valores distintos', 'Atenção / decisão']
df_dicionario = pd.DataFrame(registros)[colunas_dicionario]
assert df_dicionario.shape == (23, 11)
display(df_dicionario)

## 04. Qualidade dos dados

Foram identificadas três situações que precisam de tratamento: `PhoneService` (**serviço telefônico**) apresenta `Yes/yes` e `No/no`; `tenure` (**tempo de relacionamento**) utiliza `unknown` para indicar ausência; e `TotalCharges` (**total acumulado pago**) contém valores nulos.

### 04.1. Quantidade de registros afetados

In [0]:
# 04.1 | Quantificar as inconsistências nas bases
linhas_qa = []
for nome, base in [("Histórico", df_historico), ("Inferência", df_inferencia)]:
    linhas_qa.extend([
        {"Base":nome,"Campo":"tenure (tempo de relacionamento)",
         "Problema":"unknown (ausência disfarçada)",
         "Registros":int(base["tenure"].astype("string").str.strip().str.lower().eq("unknown").sum())},
        {"Base":nome,"Campo":"TotalCharges (valor acumulado pago)",
         "Problema":"Nulo", "Registros":int(base["TotalCharges"].isna().sum())},
        {"Base":nome,"Campo":"PhoneService (serviço telefônico)",
         "Problema":"yes/no em minúsculas",
         "Registros":int(base["PhoneService"].isin(["yes","no"]).sum())},
        {"Base":nome,"Campo":"customerID (identificador do cliente)",
         "Problema":"ID duplicado na própria base",
         "Registros":int(base["customerID"].duplicated().sum())}
    ])
df_qualidade = pd.DataFrame(linhas_qa)
display(df_qualidade)

ids_comuns = len(set(df_historico["customerID"]) & set(df_inferencia["customerID"]))
print("IDs em comum entre histórico e inferência:", ids_comuns)

### 04.2. Decisões de tratamento

**Ausências:** no histórico, apenas **um cliente** está sem `tenure` (**tempo de relacionamento**) e `TotalCharges` (**total acumulado pago**) ao mesmo tempo. Esse registro será excluído do desenvolvimento. Na inferência, não há casos assim, portanto os **1.409 clientes serão preservados**.

Quando somente uma dessas informações estiver ausente, será feita uma estimativa com as demais informações do cliente: mensalidade atual, tempo de relacionamento e total acumulado. O resultado é aproximado, pois a mensalidade pode ter mudado ao longo do contrato.

**Categorias:** `PhoneService` será padronizada para evitar que `Yes` e `yes`, por exemplo, sejam tratados como grupos diferentes. Já `No internet service` (**sem serviço de internet**) e `No phone service` (**sem serviço telefônico**) serão mantidos: essas respostas indicam que o serviço adicional não se aplica ao cliente, e não um dado ausente.

### 04.3. A ausência de TotalCharges ocorre apenas em clientes novos?

Foi investigado se os clientes sem `TotalCharges` (**total acumulado pago**) eram apenas aqueles com pouco tempo de relacionamento. Também foi comparada a taxa histórica de cancelamento entre quem tinha ou não essa informação.

In [0]:
# 04.3 | Comparar clientes com e sem TotalCharges

df_check = df_historico.copy()

df_check["tenure_num"] = pd.to_numeric(
    df_check["tenure"], errors="coerce"
)

df_check["TotalCharges_ausente"] = (
    df_check["TotalCharges"].isna()
)

resumo_ausencia = (
    df_check.groupby("TotalCharges_ausente")
    .agg(
        Clientes=("customerID", "size"),
        Tenure_mediana=("tenure_num", "median"),
        Tenure_min=("tenure_num", "min"),
        Tenure_max=("tenure_num", "max"),
        Tenure_desconhecido=("tenure_num", lambda x: x.isna().sum()),
        Churn_pct=("Churn", lambda x: 100 * x.eq("Yes").mean())
    )
    .reset_index()
)

display(resumo_ausencia.round(2))

### 04.4. Resultado da investigação

No histórico, **278 clientes** estão sem `TotalCharges` (**total acumulado pago**). Entre eles, o `tenure` (**tempo de relacionamento**) informado varia de **0 a 72 meses**, com mediana de **25 meses**. Portanto, a ausência não ocorre apenas no início do relacionamento.

A taxa observada de cancelamento nesse grupo é **33,09%**, contra **26,36%** entre os clientes com valor preenchido. Essa diferença descreve os dados, mas, por si só, não mostra que a ausência seja a causa de um cancelamento.

**Decisão:** não tratar nulo como pagamento igual a zero. Quando houver `tenure` e `MonthlyCharges` (**mensalidade atual**), o acumulado será estimado pela relação entre essas variáveis, usando `TotalCharges = tenure × MonthlyCharges`. A estimativa é aproximada porque a mensalidade pode ter variado.

## 05. Análise exploratória dos clientes

O objetivo desta etapa é identificar **quais características aparecem com maior frequência entre os clientes que cancelaram o contrato**. Os resultados mostram diferenças entre grupos, mas não significam necessariamente que uma variável tenha causado o cancelamento.

### 05.1. Frequência de cancelamentos

In [0]:
# 05.1 | Calcular a distribuição de Churn

df_churn = (
    df_historico["Churn"]
    .value_counts(dropna=False)
    .rename_axis("Churn")
    .reset_index(name="Clientes")
)

df_churn["Percentual (%)"] = (
    df_churn["Clientes"]
    / len(df_historico) * 100
).round(2)

display(df_churn)

**Resultado:** foram registrados **1.504 cancelamentos (26,7%)** entre os **5.634 clientes** históricos. Os outros **4.130 (73,3%)** não cancelaram.

Como a maior parte dos clientes não cancelou, a acurácia isolada não será suficiente para avaliar o modelo. Também será necessário verificar se ele consegue identificar e ordenar os clientes com maior risco.

### 05.2. Perfil das variáveis numéricas

Foram analisados `tenure` (**tempo de relacionamento**), `MonthlyCharges` (**mensalidade atual**), `TotalCharges` (**total acumulado pago**) e `MonthlyIncome` (**renda familiar mensal**). Para calcular as estatísticas, `unknown` em `tenure` foi tratado temporariamente como valor ausente, sem alterar os arquivos originais.

#### 05.2.1. Resumo estatístico

In [0]:
# 05.2.1 | Calcular o resumo numérico

variaveis_numericas = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "MonthlyIncome"
]

# Conversão temporária, sem modificar o histórico
df_numericas = df_historico[variaveis_numericas].apply(
    pd.to_numeric, errors="coerce"
)

resumo_numerico = df_numericas.describe().T

resumo_numerico["Nulos"] = df_numericas.isna().sum()

display(
    resumo_numerico
    .rename_axis("Variável")
    .reset_index()
    .round(2)
)

#### 05.2.2. Distribuição visual das variáveis numéricas

In [0]:
# 05.2.2 | Exibir histogramas das variáveis numéricas
import matplotlib.pyplot as plt

for coluna in variaveis_numericas:
    fig, ax = plt.subplots(figsize=(7, 3))

    ax.hist(
        df_numericas[coluna].dropna(),
        bins=30,
        edgecolor="white"
    )

    ax.set_title(f"Distribuição de {coluna}")
    ax.set_xlabel(coluna)
    ax.set_ylabel("Quantidade de clientes")

    plt.tight_layout()
    plt.show()


**Leitura das distribuições:** `tenure` (**tempo de relacionamento**) tem concentrações no início e próximo de 72 meses. `MonthlyCharges` (**mensalidade atual**) apresenta grupos de valores mais frequentes. `TotalCharges` (**total acumulado pago**) possui muitos valores baixos e poucos valores bastante altos. `MonthlyIncome` (**renda familiar mensal**) está distribuída de forma relativamente uniforme.

O total acumulado depende, entre outros fatores, de quanto o cliente pagou e por quanto tempo permaneceu. Por isso, será interpretado em conjunto com `tenure` e `MonthlyCharges`.

### 05.3. Perfil das variáveis categóricas

Foram contadas as categorias relacionadas a contrato, pagamento, serviços e perfil do cliente. A tabela permite identificar a participação de cada grupo na base.

In [0]:
# 05.3 | Calcular frequências das categorias
excluir = [
    "customerID", "Churn", "CustomerFeedback",
    "tenure", "MonthlyCharges", "TotalCharges",
    "MonthlyIncome"
]

variaveis_categoricas = [
    coluna for coluna in df_historico.columns
    if coluna not in excluir
]

# Distribuição de cada categoria
frequencias = []

for coluna in variaveis_categoricas:
    contagens = df_historico[coluna].value_counts(
        dropna=False
    )

    for categoria, quantidade in contagens.items():
        frequencias.append({
            "Variável": coluna,
            "Categoria": categoria,
            "Clientes": quantidade,
            "Percentual (%)": round(
                quantidade / len(df_historico) * 100, 2
            )
        })

df_frequencias = pd.DataFrame(frequencias)

display(df_frequencias)

**Destaques:** aproximadamente **55%** dos clientes têm contrato mensal; **44%** utilizam internet por fibra; e **22%** não contratam internet.

Quando um cliente não possui internet, os campos de serviços adicionais podem apresentar `No internet service` (**sem serviço de internet**). Essas respostas correspondem aos mesmos clientes sem internet, não a novos grupos de clientes.

### 05.4. Características categóricas e cancelamento

Foi calculada a **taxa de cancelamento dentro de cada categoria**: número de clientes que cancelaram dividido pelo total de clientes daquela categoria. Isso permite comparar grupos de tamanhos diferentes.

In [0]:
# 05.4 | Calcular taxas de cancelamento por categoria
df_biv = df_historico.copy()

# Padronização já identificada no diagnóstico
df_biv["PhoneService"] = (
    df_biv["PhoneService"].str.strip().str.capitalize()
)

# Indicador binário de cancelamento
df_biv["churn_bin"] = (df_biv["Churn"] == "Yes").astype(int)

resultados = []

for coluna in variaveis_categoricas:
    tabela = (
        df_biv.groupby(coluna, dropna=False)
        .agg(
            Clientes=("Churn", "size"),
            Cancelamentos=("churn_bin", "sum")
        )
        .reset_index()
        .rename(columns={coluna: "Categoria"})
    )

    tabela["Taxa de churn (%)"] = (
        100 * tabela["Cancelamentos"] / tabela["Clientes"]
    ).round(2)

    tabela.insert(0, "Variável", coluna)
    resultados.append(tabela)

df_churn_categoricas = pd.concat(
    resultados, ignore_index=True
)

display(
    df_churn_categoricas.sort_values(
        ["Variável", "Taxa de churn (%)"],
        ascending=[True, False]
    )
)

**Principais resultados:**

| Característica | Taxa de cancelamento observada | O que chama atenção |
|---|---|---|
| `Contract` (**tipo de contrato**) | Mensal **42,66%**; anual **11,62%**; bienal **3,27%** | Dos 1.504 cancelamentos, **1.322** ocorreram em contratos mensais. |
| `PaymentMethod` (**forma de pagamento**) | Cheque eletrônico **45,13%**; cartão automático **15,56%** | As taxas são bastante diferentes entre as formas de pagamento. |
| `InternetService` (**serviço de internet**) | Fibra **42,07%**; DSL **19,39%** | Clientes com fibra apresentam maior taxa de cancelamento no histórico. |
| `TechSupport` (**suporte técnico**) | Sem suporte **41,85%**; com suporte **15,34%** | O grupo sem suporte tem taxa de cancelamento mais elevada. |

Em `OnlineSecurity` (**segurança online**), as taxas são **41,92%** para quem não contrata o serviço e **15,02%** para quem contrata. Em `gender` (**gênero**), a diferença é pequena: **27,19%** versus **26,21%**.

**Interpretação:** essas características ajudam a identificar perfis de maior risco. No entanto, contrato, internet e serviços adicionais podem estar relacionados entre si. O modelo será utilizado para analisar as informações em conjunto, sem afirmar que uma característica isolada provoca o cancelamento.

### 05.5. Valores numéricos e cancelamento

Foram comparados os valores das variáveis numéricas entre clientes que cancelaram (`Churn = Yes`) e os que permaneceram (`Churn = No`). A mediana foi utilizada para facilitar a comparação dos valores típicos de cada grupo.

In [0]:
# 05.5 | Comparar as variáveis numéricas por Churn

df_analise = df_numericas.copy()
df_analise["Churn"] = df_historico["Churn"]

# Organizar as variáveis para comparação
df_longo = df_analise.melt(
    id_vars="Churn",
    var_name="Variável",
    value_name="Valor"
)

# Estatísticas por variável e grupo
resumo_churn = (
    df_longo
    .groupby(["Variável", "Churn"], as_index=False)
    .agg(
        Clientes=("Valor", "size"),
        Valores_validos=("Valor", "count"),
        Media=("Valor", "mean"),
        Mediana=("Valor", "median"),
        Q1=("Valor", lambda x: x.quantile(0.25)),
        Q3=("Valor", lambda x: x.quantile(0.75))
    )
)

resumo_churn["Ausentes"] = (
    resumo_churn["Clientes"] - resumo_churn["Valores_validos"]
)

display(resumo_churn.round(2))

**Resultado da comparação:**

| Variável | Mediana: cancelou | Mediana: permaneceu | Leitura |
|---|---:|---:|---|
| `tenure` (**tempo de relacionamento**) | **10 meses** | **37 meses** | Quem cancelou apresentava menor tempo de relacionamento. |
| `MonthlyCharges` (**mensalidade atual**) | **79,82** | **64,35** | Quem cancelou tinha mensalidade mais alta. |
| `TotalCharges` (**total acumulado pago**) | **753,15** | **1.672,25** | O acumulado menor é compatível com a permanência mais curta. |
| `MonthlyIncome` (**renda familiar mensal**) | **5.214,50** | **5.048,00** | As medianas são próximas. |

**Interpretação:** os clientes que cancelaram apresentam, em geral, menor tempo de relacionamento e mensalidades mais elevadas. O acumulado menor pode estar relacionado ao menor período de permanência, não necessariamente a um serviço mais barato.

Vale observar que, para clientes que cancelaram, `tenure` é medido até o cancelamento. Isso limita a interpretação do dado como uma característica disponível muitos meses antes do evento.

## 06. Limites dos dados para a previsão

### 06.1. O que significa Churn nesta base?

`Churn = Yes` indica que o cliente **cancelou o contrato**. Não significa atraso de pagamento nem fim do prazo contratual. Com esse histórico, é possível treinar um modelo de classificação e avaliar se ele diferencia os clientes que cancelaram daqueles que permaneceram.

Porém, não há datas de referência ou de cancelamento para saber **com quanta antecedência** uma previsão poderia ser feita. Assim, o desempenho nos dados disponíveis não deve ser apresentado como capacidade comprovada de prever cancelamentos nos próximos 30, 60 ou 90 dias.

### 06.2. Cuidados com o momento em que as informações foram registradas

| Informação | O que sabemos | Cuidado na modelagem |
|---|---|---|
| `CustomerFeedback` (**relato do cliente**) | O texto pode ter sido registrado após o cancelamento. | **Não utilizar no modelo principal**, pois pode revelar um evento já ocorrido. |
| `tenure` (**tempo de relacionamento**) | Para quem cancelou, corresponde ao tempo até o cancelamento; para quem permaneceu, ao tempo até a referência da base. | As informações podem representar momentos diferentes para os dois grupos. |
| `MonthlyCharges` (**mensalidade atual**) | Para quem cancelou, corresponde à última mensalidade antes do cancelamento. | Pode representar um período muito próximo do evento. |
| `TotalCharges` (**total acumulado pago**) | Reflete o acumulado até a referência disponível no registro. | Para uma previsão futura, deveria refletir somente valores conhecidos até a data da pontuação. |
| `Contract` (**tipo de contrato**) | A base contém a modalidade informada no registro, sem histórico de mudanças. | Não permite recuperar qual modalidade o cliente tinha em meses anteriores. |

**Consequência:** a divisão aleatória entre treino e teste permite avaliar o modelo nesta amostra histórica, mas não substitui uma validação temporal. O processamento também será ajustado apenas com os dados de treinamento para evitar que informações do teste sejam usadas no ajuste.

## 07. Síntese da análise exploratória

A base contém **5.634 clientes históricos**, dos quais **1.504 cancelaram**, e **1.409 clientes para inferência**. Foram identificados **19 valores `unknown` em `tenure`** (**tempo de relacionamento**) e **278 nulos em `TotalCharges`** (**total acumulado pago**) no histórico. Na inferência, são **4** e **64**, respectivamente.

As maiores diferenças observadas nas taxas de cancelamento estão relacionadas ao **tipo de contrato, forma de pagamento e serviço de internet**. Entre as variáveis numéricas, o **tempo de relacionamento** e a **mensalidade atual** também distinguem os grupos.

Esses resultados orientam a modelagem. `CustomerFeedback` permanece fora do modelo principal porque pode ter sido registrado após o cancelamento.

## 08. Construção e verificação da ABT

A base analítica (ABT) reúne os tratamentos e as novas variáveis criadas a partir das informações originais. As regras precisam funcionar da mesma maneira no histórico e na inferência.

| Transformação | O que representa | Como será utilizada |
|---|---|---|
| `tenure = round(TotalCharges / MonthlyCharges)` | Estimativa do tempo de relacionamento com base no total pago e na mensalidade atual. | Somente quando `tenure` estiver ausente. A estimativa pode variar do valor real se a mensalidade mudou. |
| `TotalCharges = tenure × MonthlyCharges` | Estimativa do valor acumulado a partir do tempo de relacionamento. | Somente quando `TotalCharges` estiver ausente. Não representa necessariamente o total efetivamente pago. |
| `comprometimentoRenda = MonthlyCharges / MonthlyIncome` | Percentual da renda familiar correspondente à mensalidade. | Característica candidata para o modelo. |
| `avgMonthlyCharges = TotalCharges / tenure` | Valor mensal médio estimado a partir do total acumulado. | Característica candidata; quando `tenure = 0`, é usada a mensalidade atual. |
| `evolucaoMonthlyCharges = MonthlyCharges / avgMonthlyCharges` | Comparação entre a mensalidade atual e a média estimada. | Valor acima de 1 indica mensalidade atual maior que a média; abaixo de 1, menor. Não mostra, sozinho, o motivo da diferença. |

A variável de evolução pode ficar igual a 1 quando `TotalCharges` foi calculado a partir de `tenure × MonthlyCharges`. Nesse caso, o resultado reflete a própria fórmula de preenchimento, e não uma evolução observada.

Não foi incluída uma mediana de `tenure` por tipo de contrato. Caso seja incluída, o cálculo deverá utilizar apenas o conjunto de treinamento. Também não foram estimados ciclos ou meses restantes de contrato, pois a base não mostra mudanças anteriores de modalidade.

`CustomerFeedback` e as classificações derivadas de seu texto não serão usados no modelo principal.

### 08.1. Conferência das ausências antes da preparação

In [0]:
# 08.1 | Verificar as ausências antes da ABT
import numpy as np

def diagnostico_ausencias(base, nome):
    t = pd.to_numeric(base['tenure'], errors='coerce')
    total = pd.to_numeric(base['TotalCharges'], errors='coerce')
    return {'Base': nome, 'Clientes originais': len(base), 'tenure ausente': int(t.isna().sum()),
            'TotalCharges ausente': int(total.isna().sum()), 'Ambos ausentes': int((t.isna() & total.isna()).sum()),
            'IDs duplicados': int(base['customerID'].duplicated().sum())}

qa_origem = pd.DataFrame([diagnostico_ausencias(df_historico, 'Histórico'),
                          diagnostico_ausencias(df_inferencia, 'Inferência')])
display(qa_origem)
assert len(df_historico) == 5634 and len(df_inferencia) == 1409, 'Volumetria diferente da base auditada.'
assert qa_origem['Ambos ausentes'].tolist() == [1, 0], 'Revisar os registros simultaneamente ausentes.'
assert qa_origem['IDs duplicados'].sum() == 0, 'Existem IDs duplicados.' 

### 08.2. Preparação das variáveis

Os valores de `Yes` e `No` são padronizados nas variáveis categóricas, para que diferenças de maiúsculas e minúsculas não criem grupos duplicados. As ausências de `tenure` (**tempo de relacionamento**) e `TotalCharges` (**total acumulado pago**) são estimadas pelas mesmas fórmulas no histórico e na inferência.

O único registro histórico sem as duas informações é excluído. Essa situação não ocorre na base de inferência recebida.

In [0]:
# 08.2 | Construir a ABT nas duas bases
def preparar_abt(base, excluir_ambos=False):
    df = base.copy()
    # Padronização das categorias, preservando ID, target e feedback original
    excluir_lower = {'customerID', 'tenure', 'TotalCharges', 'CustomerFeedback', 'Churn'}
    for col in df.select_dtypes(include=['object', 'string', 'category']).columns:
        if col not in excluir_lower:
            df[col] = df[col].astype('string').str.strip().str.lower().astype('object')
    df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'no', 1: 'yes'})
    df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce')
    for col in ['TotalCharges', 'MonthlyCharges', 'MonthlyIncome']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    ambos = df['tenure'].isna() & df['TotalCharges'].isna()
    if excluir_ambos:
        df = df.loc[~ambos].copy()
    elif ambos.any():
        raise ValueError('Há clientes sem tenure e TotalCharges; revisar cobertura antes de pontuar.')

    # Estimativas condicionais: apenas quando uma das informações está disponível
    t_ausente = df['tenure'].isna() & df['TotalCharges'].notna() & df['MonthlyCharges'].gt(0)
    df.loc[t_ausente, 'tenure'] = np.rint(df.loc[t_ausente, 'TotalCharges'] / df.loc[t_ausente, 'MonthlyCharges'])
    total_ausente = df['TotalCharges'].isna() & df['tenure'].notna() & df['MonthlyCharges'].notna()
    df.loc[total_ausente, 'TotalCharges'] = df.loc[total_ausente, 'tenure'] * df.loc[total_ausente, 'MonthlyCharges']

    df['comprometimentoRenda'] = df['MonthlyCharges'] / df['MonthlyIncome'].replace(0, np.nan)
    df['avgMonthlyCharges'] = np.where(df['tenure'].eq(0), df['MonthlyCharges'],
                                      df['TotalCharges'] / df['tenure'].replace(0, np.nan))
    df['evolucaoMonthlyCharges'] = df['MonthlyCharges'] / pd.Series(df['avgMonthlyCharges'], index=df.index).replace(0, np.nan)
    return df.replace([np.inf, -np.inf], np.nan)

df_abt_hist = preparar_abt(df_historico, excluir_ambos=True)
df_abt_inf = preparar_abt(df_inferencia, excluir_ambos=False)
print('ABT histórico:', df_abt_hist.shape, '| ABT inferência:', df_abt_inf.shape)
assert len(df_abt_hist) == 5633 and len(df_abt_inf) == 1409
assert df_abt_inf['customerID'].tolist() == df_inferencia['customerID'].tolist(), 'Ordem/IDs da inferência alterados.' 

### 08.3. Conferência das variáveis da ABT

In [0]:
# 08.3 | Conferir o resultado da ABT
features_abt = [c for c in df_abt_hist.columns if c not in ['customerID', 'Churn', 'CustomerFeedback']]
assert features_abt == [c for c in df_abt_inf.columns if c not in ['customerID', 'CustomerFeedback']]
qa_abt = pd.DataFrame({
    'Variável': features_abt,
    'Nulos histórico': [int(df_abt_hist[c].isna().sum()) for c in features_abt],
    'Nulos inferência': [int(df_abt_inf[c].isna().sum()) for c in features_abt],
})
print('Variáveis preditoras:', len(features_abt), '| Histórico:', len(df_abt_hist), '| Inferência:', len(df_abt_inf))
print('tenure estimado acima de 72 meses no histórico:', int(df_abt_hist['tenure'].gt(72).sum()))
print('tenure estimado acima de 72 meses na inferência:', int(df_abt_inf['tenure'].gt(72).sum()))
pendencias_abt = qa_abt.loc[(qa_abt['Nulos histórico'] > 0) | (qa_abt['Nulos inferência'] > 0)]
if pendencias_abt.empty:
    print('Conferência aprovada: nenhuma variável preditora apresenta valores ausentes após o tratamento.')
else:
    display(pendencias_abt)
display(df_abt_hist[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'comprometimentoRenda',
                     'avgMonthlyCharges', 'evolucaoMonthlyCharges']].head(5))
assert qa_abt[['Nulos histórico', 'Nulos inferência']].to_numpy().sum() == 0, 'Revisar as transformações: ainda existem nulos.' 

### 08.4. Resultado das conferências

A verificação confirma a quantidade de registros, procura valores ausentes que permaneceram após o tratamento e examina as variáveis calculadas.

A estimativa de `tenure` (**tempo de relacionamento**) pode ultrapassar os 72 meses observados nos valores originais. Isso ocorre porque a mensalidade atual pode ser diferente da média paga ao longo do contrato. Esses casos são identificados, mas não são corrigidos automaticamente sem informação adicional.

## 09. Roteiro da modelagem

A modelagem foi organizada em oito etapas. Os tratamentos e critérios de seleção são documentados antes da comparação dos modelos. A seção 20 consolida as principais decisões metodológicas.

| Etapa | Pergunta | Seção |
|---|---|---|
| 1. Separar as bases | Quais clientes serão usados para treino e avaliação? | 10.1 |
| 2. Selecionar variáveis por poder | Quais variáveis separam quem cancela de quem fica? | 10.2 |
| 3. Remover redundâncias | Há variáveis que repetem a mesma informação? | 10.3 |
| 4. Definir a régua | Como sabemos se um modelo é bom? | 11 |
| 5. Comparar e ajustar modelos | Algum modelo supera a regressão logística? Qual a regularização certa? | 12 |
| 6. Definir o corte | A partir de qual probabilidade o cliente vira `Yes`? | 13 |
| 7. Avaliação final e calibração | O modelo funciona em clientes nunca vistos? O score corresponde ao risco real? | 14 |
| 8. Explicar, entregar e monitorar | Quais variáveis pesam, como usar o resultado e a base nova é parecida? | 15 a 17 |

**Separação entre desenvolvimento e avaliação:** a seleção de variáveis, a escolha do algoritmo, os parâmetros e o limite de classificação utilizam apenas os dados de desenvolvimento. O conjunto de teste permanece reservado para a avaliação final.


## 10. Bases e variáveis

### 10.1. Divisão das bases

**O que fazemos:** separamos os 5.633 clientes em duas partes.

- **Treino (80%)** — o modelo aprende aqui. Dentro dele usamos **validação cruzada em 5 partes** (*5-fold cross-validation*): o treino é dividido em 5 blocos; o modelo treina em 4 e é avaliado no 5º, girando até cada bloco ter sido avaliado uma vez. Assim cada cliente do treino recebe uma nota "de fora", dada por um modelo que não o viu. É com essas notas que escolhemos modelo e corte.
- **Teste (20%)** — reservado para avaliar o modelo selecionado, sem participar da escolha das variáveis, do algoritmo ou do corte.

**Estratificada** quer dizer que as duas partes mantêm a mesma proporção de cancelamentos (~26,7%), para que nenhuma fique artificialmente mais fácil.

**Variável adicional:** `TotalCharges_ausente` (1 = o total pago veio vazio no arquivo e foi estimado). Na seção 04, esse grupo teve 33% de churn contra 26% do restante; a marcação permite que o modelo use essa diferença e deixa claro quais totais são estimados.

Por que não usar datas? A base não tem datas (seção 06). A divisão aleatória é a melhor disponível, e essa limitação é declarada.

| Parâmetro | Valor | Por quê |
|---|---|---|
| Tamanho do teste (`test_size`) | 20% | Deixa ~1.100 clientes (~300 cancelamentos) para a avaliação final, suficiente para medir AUC e decis com estabilidade, sem tirar demais do treino. 80/20 é o padrão de mercado para bases deste tamanho. |
| Estratificação (`stratify=y`) | sim | Garante a mesma taxa de churn nas duas partes. Sem isso, o acaso poderia deixar o teste mais fácil ou mais difícil. |
| Partes da validação cruzada (`n_splits`) | 5 | Em cada rodada, 80% do conjunto de desenvolvimento é usado para ajuste e 20% para validação. Cinco divisões estratificadas oferecem repetição da avaliação e custo computacional moderado, mantendo proporção semelhante de cancelamentos nos grupos. |
| Semente aleatória (`random_state`) | 42 | Fixa o sorteio para que qualquer pessoa obtenha exatamente os mesmos números ao rodar o notebook (reprodutibilidade). O valor em si não importa. |


In [0]:
# 10 | Separar treino e teste
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, roc_curve, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

X = df_abt_hist[features_abt].copy()
y = df_abt_hist['Churn'].astype('string').str.strip().str.lower().map({'yes': 1, 'no': 0}).astype(int)
X_inferencia = df_abt_inf[features_abt].copy()

# Indicador de que TotalCharges veio vazio no arquivo original (o valor foi estimado na ABT)
X['TotalCharges_ausente'] = df_historico.loc[X.index, 'TotalCharges'].isna().astype(int)
X_inferencia['TotalCharges_ausente'] = df_inferencia.loc[X_inferencia.index, 'TotalCharges'].isna().astype(int)

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

display(pd.DataFrame({
    'Base': ['Treino (80%)', 'Teste (20%)', 'Total'],
    'Clientes': [len(y_treino), len(y_teste), len(y)],
    'Cancelamentos': [int(y_treino.sum()), int(y_teste.sum()), int(y.sum())],
    'Taxa de churn (%)': [round(100*y_treino.mean(), 2), round(100*y_teste.mean(), 2), round(100*y.mean(), 2)]
}))


**Leitura:** treino e teste têm a mesma taxa de cancelamento (~26,7%). A divisão não favoreceu nenhum lado.


### 10.2. Seleção de variáveis por Information Value (IV)

Até aqui entram 24 candidatas: as 23 da ABT e o indicador de total ausente. Nem toda variável ajuda: algumas não separam quem cancela de quem fica e só acrescentam ruído e coisas a explicar.

O **IV** (*Information Value*, valor de informação) é a medida padrão em modelos de risco para avaliar uma variável sozinha. Para cada faixa (ou categoria) da variável, compara a proporção de quem **cancelou** com a de quem **ficou**:

`IV = Σ (% dos que ficaram − % dos que cancelaram) × ln(% dos que ficaram / % dos que cancelaram)`

Se as proporções são iguais em todas as faixas, a variável não diferencia nada e o IV é zero. Variáveis numéricas são cortadas em 10 faixas de mesmo tamanho.

| IV | Leitura de mercado |
|---|---|
| abaixo de 0,02 | capacidade individual de discriminação muito baixa → candidata à exclusão |
| 0,02 a 0,10 | fraco |
| 0,10 a 0,30 | médio |
| acima de 0,30 | forte |

**Regras:** o IV é calculado **só no treino** (o teste continua trancado); o corte de 0,02 foi definido antes de olhar o resultado. Para confirmar que a remoção não custa desempenho, comparamos a AUC na validação cruzada com todas as candidatas e só com as selecionadas.


In [0]:
# 10.2 | Information Value no treino e seleção (IV >= 0,02)
def information_value(serie, alvo):
    if pd.api.types.is_numeric_dtype(serie) and serie.nunique() > 10:
        serie = pd.qcut(serie, 10, duplicates='drop')
    tab = pd.crosstab(serie, alvo).clip(lower=0.5)          # 0,5 evita divisão por zero em faixas vazias
    pct_cancelou = tab[1] / tab[1].sum(); pct_ficou = tab[0] / tab[0].sum()
    return float(((pct_ficou - pct_cancelou) * np.log(pct_ficou / pct_cancelou)).sum())

def forca_iv(v): return 'sem poder' if v < 0.02 else 'fraco' if v < 0.10 else 'médio' if v < 0.30 else 'forte'
tabela_iv = pd.DataFrame({'Variável': X_treino.columns,
                          'IV': [information_value(X_treino[c], y_treino) for c in X_treino.columns]})
tabela_iv['Força'] = tabela_iv['IV'].apply(forca_iv)
tabela_iv['Decisão'] = np.where(tabela_iv['IV'] >= 0.02, 'mantém', 'sai')
if not (tabela_iv['Decisão'] == 'mantém').any():
    # Sem sinal individual no IV, não é adequado treinar um modelo sem preditoras.
    tabela_iv['Decisão'] = 'mantém'
    print('Nenhuma variável alcançou IV 0,02: o filtro foi suspenso e as candidatas foram preservadas para avaliação.')
tabela_iv = tabela_iv.sort_values('IV', ascending=False).reset_index(drop=True)
display(tabela_iv.round(3))

cores = tabela_iv['Decisão'].map({'mantém': '#2F6FB5', 'sai': '#C9CED6'})[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(tabela_iv['Variável'][::-1], tabela_iv['IV'][::-1], color=cores)
ax.axvline(0.02, ls='--', color='black', lw=1); ax.text(0.03, 0.5, 'corte 0,02', fontsize=9)
ax.set_xlabel('Information Value (calculado no treino)'); ax.set_title('Poder de cada variável para separar quem cancela')
plt.tight_layout(); plt.show()

# Confirmar que a remoção não custa desempenho (mesma regressão logística, validação cruzada no treino)
def auc_cv_colunas(colunas):
    cat = [c for c in colunas if not pd.api.types.is_numeric_dtype(X_treino[c])]; num = [c for c in colunas if c not in cat]
    prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), cat),
                              ('num', StandardScaler(), num)])
    r = cross_validate(Pipeline([('p', prep), ('m', LogisticRegression(max_iter=3000))]),
                       X_treino[colunas], y_treino, cv=cv, scoring='roc_auc')['test_score']
    return r.mean(), r.std()
selecionadas = tabela_iv.loc[tabela_iv['Decisão'] == 'mantém', 'Variável'].tolist()
removidas = tabela_iv.loc[tabela_iv['Decisão'] == 'sai', 'Variável'].tolist()
auc_todas, dp_todas = auc_cv_colunas(list(X_treino.columns)); auc_sel, _ = auc_cv_colunas(selecionadas)
display(pd.DataFrame({'Versão': [f'Todas as candidatas ({X_treino.shape[1]})', f'Selecionadas por IV ({len(selecionadas)})'],
                      'AUC validação': [auc_todas, auc_sel]}).round(4))
print(f'Removidas: {removidas}')
print(f'Diferença de AUC: {auc_todas - auc_sel:.4f} | variação natural entre rodadas: {dp_todas:.4f}')

# Guardar a base completa (usada na seção 15.4) e seguir só com as selecionadas
X_treino_completo, X_teste_completo = X_treino.copy(), X_teste.copy()
X, X_treino, X_teste, X_inferencia = X[selecionadas], X_treino[selecionadas], X_teste[selecionadas], X_inferencia[selecionadas]
print(f'Variáveis no modelo a partir daqui: {len(selecionadas)}')


**Leitura:** **19 das 24 candidatas ficam no modelo.** Saem cinco variáveis com IV abaixo do limite adotado (IV abaixo de 0,02): **gênero** (0,000), **serviço telefônico** (0,000), **total ausente** (0,004), **renda mensal** (0,008) e **múltiplas linhas** (0,013). Isso bate com a análise exploratória: gênero tinha 27,2% contra 26,2% de churn, e a renda tinha medianas quase iguais entre quem cancelou e quem ficou.

**A remoção não custou desempenho:** a AUC passa de 0,840 para 0,838. A diferença (0,001) é muito menor que a variação natural entre rodadas (0,018). Com o mesmo resultado, ficamos com o modelo mais enxuto, com menos variáveis para explicar, monitorar e manter.

**Dois cuidados de leitura:**
- A renda sai sozinha, mas continua no modelo **dentro do comprometimento de renda** (IV 0,16), que é a forma em que ela tem sentido de negócio.
- As variáveis de serviços adicionais (segurança, suporte, backup) têm IV alto em parte porque incluem a categoria "sem internet", um grupo que quase não cancela (7%). É informação real, não erro.


### 10.3. Redundância entre variáveis (correlação)

**Por que esta etapa:** o IV avalia cada variável **sozinha**. Duas variáveis podem ter IV alto e carregar exatamente a mesma informação. Na regressão logística isso não piora a previsão, mas o modelo divide o peso entre as duas de forma arbitrária, e os pesos deixam de ser interpretáveis. É a chamada **multicolinearidade**.

**Como medimos:** a **correlação de Pearson** entre cada par de variáveis numéricas, calculada só no treino. Vai de −1 a 1: perto de ±1, as duas sobem e descem juntas; perto de 0, não têm relação linear.

**Critério adotado:** correlação absoluta acima de **0,90** indica que duas variáveis numéricas podem estar trazendo informações praticamente repetidas. No critério adotado, **quando o par contém uma variável original e uma derivada, é retirada a derivada e mantida a original**, por ser mais simples de explicar e reproduzir. Em seguida, comparamos a AUC da validação cruzada antes e depois da exclusão para verificar se houve perda de desempenho. Correlações entre 0,70 e 0,90 são apenas sinalizadas, sem exclusão automática.

**Exemplo da base:** `MonthlyCharges` × `avgMonthlyCharges` apresentam correlação de **0,997**. Foi mantida a mensalidade atual (`MonthlyCharges`) e retirada a média calculada (`avgMonthlyCharges`). A AUC ficou praticamente igual (**0,838** antes e depois).

**Limite desta regra:** o código não exclui automaticamente um par formado por duas variáveis originais. Se aparecerem duas variáveis derivadas muito correlacionadas, a decisão exige revisão, pois a implementação atual não compara suas contribuições. Nesses casos, IV, significado de negócio e AUC/KS na validação devem orientar a escolha. A comparação de AUC apresentada aqui é uma **verificação posterior**, não uma seleção automática que desfaz exclusões. O valor 0,90 é um critério operacional conservador, não uma regra estatística universal.

**Limite:** a correlação só mede relação linear entre numéricas. Para as categóricas, a redundância foi tratada na análise exploratória e pela categoria "sem internet", que é a mesma em vários campos de serviço.


In [0]:
# 10.3 | Correlação entre variáveis numéricas (treino) e remoção de redundâncias
num_sel = [c for c in X_treino.columns if pd.api.types.is_numeric_dtype(X_treino[c])]
corr = X_treino[num_sel].corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_sel)), num_sel, rotation=45, ha='right'); ax.set_yticks(range(len(num_sel)), num_sel)
for i in range(len(num_sel)):
    for j in range(len(num_sel)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if abs(corr.iloc[i, j]) > 0.6 else 'black')
fig.colorbar(im, ax=ax, shrink=0.8); ax.set_title('Correlação entre variáveis numéricas (treino)')
plt.tight_layout(); plt.show()

derivadas = ['avgMonthlyCharges', 'evolucaoMonthlyCharges', 'comprometimentoRenda']
pares = []
for i, a in enumerate(num_sel):
    for b in num_sel[i+1:]:
        r_ab = corr.loc[a, b]
        if abs(r_ab) >= 0.70:
            sai = (b if b in derivadas else a if a in derivadas else '—') if abs(r_ab) > 0.90 else '—'
            pares.append({'Variável A': a, 'Variável B': b, 'Correlação': round(r_ab, 3),
                          'Situação': 'redundante (> 0,90)' if abs(r_ab) > 0.90 else 'relacionadas (0,70–0,90)',
                          'Sai do modelo': sai})
tabela_pares = pd.DataFrame(pares, columns=['Variável A', 'Variável B', 'Correlação', 'Situação', 'Sai do modelo'])
if tabela_pares.empty:
    print('Nenhum par acima de 0,70; não há exclusões por redundância.')
else:
    display(tabela_pares)
remover_corr = sorted({p for p in tabela_pares['Sai do modelo'] if p != '—'}) if len(tabela_pares) else []

antes = list(X_treino.columns); depois = [c for c in antes if c not in remover_corr]
auc_antes, dp_antes = auc_cv_colunas(antes); auc_depois, _ = auc_cv_colunas(depois)
display(pd.DataFrame({'Versão': [f'Após IV ({len(antes)})', f'Após correlação ({len(depois)})'],
                      'AUC validação': [auc_antes, auc_depois]}).round(4))
print(f'Removidas por redundância: {remover_corr} | variação natural entre rodadas: {dp_antes:.4f}')
X, X_treino, X_teste, X_inferencia = X[depois], X_treino[depois], X_teste[depois], X_inferencia[depois]
print(f'Variáveis finais no modelo: {len(depois)}'); print(depois)


**Leitura:** **Como ler o mapa:** cada quadrado é um par de variáveis. Vermelho escuro = sobem juntas; azul = uma sobe quando a outra desce; claro = sem relação linear. A diagonal é sempre 1 (cada variável com ela mesma).

**Resultado:**
- **Mensalidade média × mensalidade atual: 0,997.** São praticamente a mesma variável: na prática a mensalidade quase não mudou ao longo dos contratos, então total ÷ meses devolve a própria mensalidade. Pela regra, sai a derivada (**`avgMonthlyCharges`**) e fica a original (`MonthlyCharges`).
- **Tempo de casa × total pago: 0,82.** Relação forte e esperada (total ≈ meses × mensalidade), mas abaixo de 0,90: as duas ficam, porque o total também carrega o valor da mensalidade.
- **Evolução da mensalidade e comprometimento de renda** têm correlação abaixo de 0,25 com todas as outras: trazem informação própria e ficam.

**Conferência após a exclusão:** a AUC da validação cruzada ficou em torno de 0,838 antes e depois. A exclusão não apresentou perda relevante de capacidade de discriminação nessa comparação. **O modelo final tem 18 variáveis.**


### 10.4. Pipeline de preparação e modelagem

O pipeline recebe as colunas originais e executa, nesta ordem: **padronização e imputação na ABT → variáveis derivadas → seleção das 18 variáveis aprovadas no treino → codificação das categorias → escala das numéricas → algoritmo**. As etapas que aprendem parâmetros durante o ajuste (codificação e escala) são treinadas dentro de cada rodada da validação cruzada e reutilizadas na previsão.

A seleção por IV e correlação foi definida anteriormente, **somente com o conjunto de desenvolvimento**, e fica congelada para o modelo final. Nesta implementação, IV e correlação são calculados no conjunto de desenvolvimento antes das rodadas de validação cruzada. Uma seleção totalmente interna a cada rodada exigiria seletores ajustáveis no pipeline. O conjunto de teste permanece independente dessa seleção.

In [0]:
# 10.4 | Pipeline reutilizável: tratamentos, seleção, codificação e modelo
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Campos necessários para pontuar novos clientes, sem ID, resposta ou feedback.
COLUNAS_ENTRADA = [c for c in df_inferencia.columns
                   if c not in ['customerID', 'CustomerFeedback', 'Churn']]
COLUNAS_MODELO = X.columns.tolist()  # 18 variáveis aprovadas na seleção da seção 10

class EngenhariaABT(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # O tratamento por registro não aprende estatísticas globais.
        return self

    def transform(self, X):
        base = X.copy()
        # A ausência original é um indicador informativo antes da estimativa.
        falta_total = pd.to_numeric(base['TotalCharges'], errors='coerce').isna().astype(int)
        base = preparar_abt(base)
        base['TotalCharges_ausente'] = falta_total
        return base

class SelecionarVariaveis(BaseEstimator, TransformerMixin):
    def __init__(self, colunas):
        self.colunas = colunas

    def fit(self, X, y=None):
        faltantes = [c for c in self.colunas if c not in X.columns]
        if faltantes:
            raise ValueError(f'Variáveis esperadas não encontradas: {faltantes}')
        return self

    def transform(self, X):
        return X.loc[:, self.colunas].copy()


# Seleção feita na seção 10 usando somente o conjunto de desenvolvimento.
# O conjunto escolhido fica congelado para avaliar, reestimar e pontuar novos clientes.
cols_cat = X.select_dtypes(include=['object', 'string']).columns.tolist()
cols_num = [c for c in X.columns if c not in cols_cat]
preprocessamento = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), cols_cat),
    ('num', StandardScaler(), cols_num)
])

def construir_pipeline(modelo):
    return Pipeline([
        ('abt', EngenhariaABT()),
        ('selecao', SelecionarVariaveis(COLUNAS_MODELO)),
        ('preparacao', clone(preprocessamento)),
        ('modelo', clone(modelo))
    ])

pipeline_base = construir_pipeline(LogisticRegression(max_iter=3000, random_state=42))
X_bruto_historico = df_historico.loc[df_abt_hist.index, COLUNAS_ENTRADA].copy()
X_bruto_inferencia = df_inferencia[COLUNAS_ENTRADA].copy()
X_bruto_treino = X_bruto_historico.loc[X_treino.index]
X_bruto_teste = X_bruto_historico.loc[X_teste.index]
assert len(X_bruto_historico) == 5633 and len(X_bruto_inferencia) == 1409
print(f'Entrada: {len(COLUNAS_ENTRADA)} campos originais | Preditoras selecionadas: {len(COLUNAS_MODELO)}')

### 10.5. Registro dos experimentos

O MLflow registra os algoritmos comparados, as métricas das rodadas de validação e os parâmetros testados. Cada execução utiliza um **experimento ativo próprio do projeto**, separado do experimento associado ao notebook, para manter o histórico disponível mesmo após renomeações ou importações do arquivo.

O registro do pipeline final ocorre na seção 18, depois da avaliação e das previsões.


In [0]:
# 10.5 | Criar/selecionar um experimento ativo e registrar a comparação no MLflow
import os
import logging
from datetime import datetime
os.environ['MLFLOW_DATABRICKS_TAGS_TO_SKIP'] = 'extraContext'
logging.getLogger('mlflow.tracking.context.registry').setLevel(logging.ERROR)
REGISTROS_MLFLOW = []
MLFLOW_DISPONIVEL = False
MLFLOW_EXPERIMENTO_ID = None
MLFLOW_EXPERIMENTO_NOME = None

try:
    import mlflow
    import mlflow.sklearn
    usuario_experimento = spark.sql('SELECT current_user()').first()[0]
    # Um experimento do Workspace por usuário, sem caminho pessoal fixado no código.
    nome_base = f'/Users/{usuario_experimento}/Terra_Signal_Churn_Grupo6'
    anterior = mlflow.get_experiment_by_name(nome_base)
    if anterior is not None and anterior.lifecycle_stage != 'active':
        # Evita reutilizar um experimento já enviado à lixeira.
        nome_base += '_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    experimento = mlflow.set_experiment(experiment_name=nome_base)
    if experimento.lifecycle_stage != 'active':
        raise RuntimeError(f'Experimento MLflow não está ativo: {experimento.lifecycle_stage}')
    MLFLOW_EXPERIMENTO_ID = str(experimento.experiment_id)
    MLFLOW_EXPERIMENTO_NOME = experimento.name
    MLFLOW_DISPONIVEL = True
    print(f'MLflow ativo | Experimento: {MLFLOW_EXPERIMENTO_NOME} | ID: {MLFLOW_EXPERIMENTO_ID}')
except Exception as exc:
    print(f'MLflow não configurado: {type(exc).__name__}: {exc}')
    print('O treinamento e as previsões continuam disponíveis; os registros devem ser refeitos após corrigir o acesso.')

def registrar_resultado_mlflow(nome, metricas, parametros):
    if not MLFLOW_DISPONIVEL:
        REGISTROS_MLFLOW.append({'Execução': nome, 'Situação': 'Pendente', 'run_id': None})
        return None
    try:
        with mlflow.start_run(experiment_id=MLFLOW_EXPERIMENTO_ID, run_name=nome) as run:
            mlflow.log_params({str(k): str(v) for k, v in parametros.items()})
            mlflow.log_metrics({str(k): float(v) for k, v in metricas.items()})
            REGISTROS_MLFLOW.append({'Execução': nome, 'Situação': 'Registrado',
                                     'run_id': run.info.run_id, 'experiment_id': run.info.experiment_id})
            return run.info.run_id
    except Exception as exc:
        REGISTROS_MLFLOW.append({'Execução': nome, 'Situação': 'Pendente', 'run_id': None})
        print(f'Registro MLflow pendente [{nome}]: {type(exc).__name__}: {exc}')
        return None


## 11. Critérios de avaliação do modelo de churn

O modelo não responde "vai cancelar" diretamente. Ele dá a cada cliente uma **probabilidade de cancelar** (o *score*). O uso de negócio é **ordenar**: ligar primeiro para quem tem score mais alto. Por isso as métricas principais medem ordenação, e as de classificação só entram depois do corte.

| Métrica | O que responde | Como ler | Referência prática |
|---|---|---|---|
| **AUC** (*Area Under the ROC Curve*, área sob a curva ROC) | Sorteando um cliente que cancelou e um que ficou, qual a chance de o modelo dar score maior ao que cancelou? | 0,5 = sorteio; 1,0 = perfeito | 0,70–0,80 bom; acima de 0,80 muito bom |
| **Gini** | Mesma informação da AUC, na escala usada em crédito | Gini = 2 × AUC − 1 | acima de 0,60 muito bom |
| **KS** (*Kolmogorov-Smirnov*) | Qual a maior distância entre as curvas acumuladas de score de quem cancelou e de quem ficou? | 0 = não separa; 1 = separa tudo | acima de 0,40 bom |
| **Taxa de churn por decil** | Os 10% de maior score cancelam mais do que os 10% seguintes, e assim por diante? | tende a crescer do D1 (menor score) ao D10 (maior score) | queda contínua, sem inversões |
| **Lift** | Quantas vezes a taxa de churn do grupo de maior risco supera a média | lift 2 = o dobro da média | acima de 2 no top 10% |

Depois do corte (seção 13), três métricas de classificação:

| Métrica | Pergunta de negócio |
|---|---|
| **Precisão** (*precision*) | Dos clientes marcados `Yes`, quantos de fato cancelaram? (desperdício de ligação) |
| **Recall** (sensibilidade) | Dos clientes que cancelaram, quantos o modelo marcou? (churn que escapa) |
| **F1** | Média que equilibra precisão e recall |

### 11.1. Exemplo ilustrativo de classificação

Para ilustrar as métricas, considere **100 clientes**, dos quais **40 foram classificados como `Yes`**. No resultado observado, **27 cancelaram o contrato**.

| | Cancelou de fato | Ficou de fato |
|---|---:|---:|
| **Marcado `Yes`** | 21 — acerto (*verdadeiro positivo*) | 19 — ligação sem necessidade (*falso positivo*) |
| **Marcado `No`** | 6 — cancelamento que escapou (*falso negativo*) | 54 — acerto (*verdadeiro negativo*) |

- **Precisão = 21 / 40 = 52%.** Dos 40 clientes classificados como `Yes`, 21 efetivamente cancelaram. Essa métrica indica a proporção de alertas corretos.
- **Recall = 21 / 27 = 78%.** Dos 27 clientes que cancelaram, 21 foram identificados pelo modelo. Essa métrica mostra quantos cancelamentos observados foram sinalizados.
- **F1 = média entre as duas** (média harmônica, que puxa para o valor mais baixo). Só fica alto quando precisão **e** recall estão razoáveis. Serve para comparar cortes com um único número.

**O equilíbrio:** baixar o corte marca mais gente → o recall sobe (escapa menos churn) e a precisão cai (mais ligações desnecessárias). Subir o corte faz o contrário. Não existe corte que maximize os dois ao mesmo tempo.

**Por que não acurácia?** Com 73% de clientes que ficam, um modelo que responde "ninguém cancela" acerta 73% e não serve para nada.


## 12. Comparação e ajuste de modelos

### 12.0. Como os dados entram nos modelos

| Decisão | O que é | Por quê |
|---|---|---|
| **Codificação das categorias** (*one-hot encoding* com `drop='first'`) | Cada categoria vira uma coluna 0/1; a primeira categoria de cada variável é retirada e vira a **referência** (ex.: contrato mensal). | Modelos só leem números. Retirar uma categoria evita colunas que somam sempre 1 (redundância perfeita) e faz cada peso da logística significar "esta categoria comparada à referência". |
| **Padronização das numéricas** (`StandardScaler`) | Cada numérica passa a ter média 0 e desvio-padrão 1. | Tempo de casa vai até 72 e total pago até 8.600. Sem padronizar, a regularização da logística penalizaria as variáveis de forma desigual, e os pesos não seriam comparáveis entre si. Nos modelos de árvore não faz diferença, mas também não atrapalha. |
| **Categoria nova na inferência** (`handle_unknown='ignore'`) | Uma categoria que não existia no treino vira "todas as colunas 0". | Evita que o modelo quebre em produção. |
| **Tudo dentro de um *pipeline*** | Codificação e padronização são ajustadas só com os dados de treino de cada rodada. | Se a média e o desvio fossem calculados com a base inteira, informação do teste vazaria para o treino. |
| **Desbalanceamento** (27% churn × 73% não) | **Não** usamos *oversampling* (ex.: SMOTE) nem pesos de classe. | Optou-se por manter a distribuição original das classes. Técnicas de balanceamento podem alterar o ajuste e a calibração das probabilidades; sua adoção exigiria avaliação específica. A definição do corte é tratada separadamente na seção 13. |

### 12.1. Quais modelos e por quê

| Papel | Modelo | Por que faz sentido em churn |
|---|---|---|
| **Baseline** | **Regressão logística** | Padrão de mercado em risco (crédito, churn). Gera probabilidades bem calibradas e pesos interpretáveis por variável. É a referência que os outros precisam superar. |
| Desafiante | **Árvore de decisão** | Regras simples ("contrato mensal e menos de 12 meses → alto risco"). Fácil de explicar, mas costuma ordenar pior. |
| Desafiante | **Random Forest** | Muitas árvores combinadas. Capta interações (ex.: fibra + mensal) sem precisar criá-las. |
| Desafiante | **Gradient Boosting** (*HistGradientBoosting*, do scikit-learn) | Árvores em sequência, cada uma corrigindo os erros da anterior. Costuma ser o mais forte em dados tabulares. |
| Desafiante | **XGBoost** | Mesma ideia do Gradient Boosting, na implementação mais usada no mercado e em competições. |

**Parâmetros dos desafiantes e por quê.** Com ~4.500 clientes no treino, o maior risco é o modelo decorar o treino. Todos os parâmetros abaixo **limitam a complexidade** e são valores usuais de mercado para bases deste porte. Não fizemos uma busca extensa de parâmetros nos desafiantes porque, como se vê na 12.4, eles já empatam com a logística; uma busca só os deixaria mais ajustados ao treino.

| Modelo | Parâmetros | Por quê |
|---|---|---|
| Regressão logística | `C=1` (inicial; ajustado na 12.5), `max_iter=3000` | `C` controla a regularização (quanto os pesos são "freados"). `max_iter` alto só garante que o cálculo termine. |
| Árvore de decisão | `max_depth=5`, `min_samples_leaf=50` | No máximo 5 perguntas em sequência e pelo menos 50 clientes por folha: regras legíveis e sem folhas baseadas em poucos casos. |
| Random Forest | 300 árvores, `max_depth=8`, `min_samples_leaf=20` | 300 árvores estabilizam a média. Profundidade e folha mínima evitam árvores que memorizam clientes. |
| Gradient Boosting | `max_depth=3`, `learning_rate=0.05`, 200 iterações, `min_samples_leaf=40` | Árvores rasas e aprendizado lento: cada árvore corrige um pouco do erro, o que reduz sobreajuste. |
| XGBoost | 220 árvores, `max_depth=3`, `learning_rate=0.05`, `subsample=0.85`, `colsample_bytree=0.85` | Mesma lógica; cada árvore vê 85% dos clientes e 85% das variáveis, o que dá variedade e reduz sobreajuste. |

### 12.2. Regra de decisão, definida antes de olhar o resultado

1. Critério principal: **AUC média na validação cruzada**.
2. Um desafiante só substitui a regressão logística se ganhar **mais do que a variação natural entre as 5 rodadas** (o desvio-padrão da AUC). Ganho menor que isso é ruído.
3. Critério de desempate: **diferença entre AUC no treino e na validação**. Diferença grande = o modelo decorou o treino (*overfitting*, sobreajuste) e tende a piorar em clientes novos.


**Execução:** a partir da seção 10.4, todos os algoritmos recebem os mesmos campos originais pelo pipeline; os tratamentos e o ajuste do codificador e do escalonamento são aplicados dentro de cada treinamento da validação cruzada. O MLflow registra as métricas e os parâmetros de cada alternativa, sem acessar os 20% reservados ao teste.


In [0]:
# 12.3 | Comparar modelos: mesmo pipeline desde os dados originais
modelos = {
    'Regressão logística (baseline)': LogisticRegression(max_iter=3000),
    'Árvore de decisão': DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1),
    'Gradient Boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                                        min_samples_leaf=40, random_state=42)
}
from xgboost import XGBClassifier
modelos['XGBoost'] = XGBClassifier(n_estimators=220, max_depth=3, learning_rate=0.05, subsample=0.85,
                                  colsample_bytree=0.85, eval_metric='logloss', random_state=42, n_jobs=-1)
pipelines = {nome: construir_pipeline(m) for nome, m in modelos.items()}

linhas = []
for nome, pipe in pipelines.items():
    res = cross_validate(pipe, X_bruto_treino, y_treino, cv=cv, scoring='roc_auc', return_train_score=True)
    metricas_cv = {'auc_cv': res['test_score'].mean(), 'auc_cv_dp': res['test_score'].std(),
                   'auc_treino_cv': res['train_score'].mean()}
    metricas_cv.update({f'auc_dobra_{i}': v for i, v in enumerate(res['test_score'], 1)})
    parametros_cv = {'algoritmo': nome, 'cv_folds': cv.n_splits, 'selecionadas': len(COLUNAS_MODELO),
                     'preparacao': 'pipeline_completo', 'classe_positiva': 'Churn=Yes'}
    parametros_cv.update({f'modelo_{k}': v for k, v in pipe.named_steps['modelo'].get_params(deep=False).items()})
    run_id = registrar_resultado_mlflow(f'cv_{nome}', metricas_cv, parametros_cv)
    linhas.append({'Modelo': nome,
                   'AUC validação': res['test_score'].mean(),
                   'Desvio entre rodadas': res['test_score'].std(),
                   'AUC treino': res['train_score'].mean(),
                   'Gap treino − validação': res['train_score'].mean() - res['test_score'].mean(),
                   'Registro MLflow': 'Registrado' if run_id else 'Pendente'})
df_comparacao = pd.DataFrame(linhas)
df_comparacao['Gini validação'] = 2 * df_comparacao['AUC validação'] - 1
base = df_comparacao.iloc[0]
df_comparacao['Ganho vs baseline'] = df_comparacao['AUC validação'] - base['AUC validação']
df_comparacao['Ganho supera o ruído?'] = np.where(
    df_comparacao['Ganho vs baseline'] > base['Desvio entre rodadas'], 'Sim', 'Não')
display(df_comparacao.round(4))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(df_comparacao['Modelo'], df_comparacao['AUC validação'],
        xerr=df_comparacao['Desvio entre rodadas'], color=['#E07A2E'] + ['#7FA6D6']*(len(df_comparacao)-1))
ax.set_xlim(0, min(1.0, max(0.90, df_comparacao['AUC validação'].max()+0.05))); ax.set_xlabel('AUC na validação cruzada (barra = variação entre as 5 rodadas)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()


### 12.4. Decisão

A comparação dos algoritmos é apresentada na tabela e no gráfico acima, com **AUC média na validação cruzada**, variação entre as cinco rodadas e diferença entre treino e validação. O XGBoost participa da comparação após a instalação da biblioteca.

A **regressão logística** permanece como o modelo escolhido nesta implementação. A decisão considera o desempenho observado, a diferença entre treino e validação e a facilidade de interpretar as previsões. Se o XGBoost apresentar ganho relevante após a nova execução, esse resultado deverá ser registrado e a justificativa da escolha reavaliada antes da entrega definitiva. O conjunto de teste não é utilizado para essa seleção.

### 12.5. Ajuste da regularização da regressão logística

**O que é:** a regularização "freia" os pesos para que o modelo não se apoie demais em padrões do treino que podem ser acaso. O parâmetro `C` controla esse freio: **`C` pequeno = freio forte** (pesos menores, modelo mais conservador); **`C` grande = freio fraco**.

**Por que ajustar:** até aqui usamos `C=1`, que é só o padrão do software. Testamos uma grade de valores na mesma validação cruzada (só treino) e aplicamos a mesma regra da 12.2: um valor só substitui o padrão se ganhar mais do que a variação natural entre as rodadas.


In [0]:
# 12.5 | Grade de C para a regressão logística (validação cruzada no treino)
grade_C = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
linhas_C = []
for C in grade_C:
    pipe_C = clone(pipelines['Regressão logística (baseline)']).set_params(modelo__C=C)
    r_C = cross_validate(pipe_C, X_bruto_treino, y_treino, cv=cv, scoring='roc_auc')['test_score']
    linhas_C.append({'C': C, 'AUC validação': r_C.mean(), 'Desvio': r_C.std()})
tabela_C = pd.DataFrame(linhas_C); display(tabela_C.round(4))

ref = tabela_C.loc[tabela_C['C'] == 1].iloc[0]; melhor = tabela_C.loc[tabela_C['AUC validação'].idxmax()]
C_escolhido = float(melhor['C']) if melhor['AUC validação'] - ref['AUC validação'] > ref['Desvio'] else 1.0
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.errorbar(range(len(grade_C)), tabela_C['AUC validação'], yerr=tabela_C['Desvio'], marker='o', color='#2F6FB5', capsize=3)
ax.set_xticks(range(len(grade_C)), [str(c).replace('.', ',') for c in grade_C])
ax.set_xlabel('C (← freio forte | freio fraco →)'); ax.set_ylabel('AUC validação'); ax.set_title('Efeito da regularização')
plt.tight_layout(); plt.show()
print(f'Melhor C na grade: {melhor["C"]} (AUC {melhor["AUC validação"]:.4f}) | C=1: AUC {ref["AUC validação"]:.4f} | C escolhido: {C_escolhido}')

# Registrar os sete valores de C para permitir consultar os experimentos individualmente.
status_C = []
for _, linha in tabela_C.iterrows():
    valor_C = float(linha['C'])
    run_id_C = registrar_resultado_mlflow(
        f'cv_reglog_C_{valor_C:g}',
        {'auc_cv': float(linha['AUC validação']), 'auc_cv_dp': float(linha['Desvio'])},
        {'algoritmo': 'Regressão logística', 'C': valor_C, 'cv_folds': cv.n_splits,
         'selecionado': valor_C == C_escolhido, 'base': 'treino_80pct'})
    status_C.append('Registrado' if run_id_C else 'Pendente')
tabela_C['Registro MLflow'] = status_C
print(f'Experimentos de regularização registrados: {sum(s == "Registrado" for s in status_C)} de {len(status_C)}.')


### 12.6. Histórico dos experimentos no MLflow

Cada algoritmo e cada valor testado de `C` tem sua execução registrada, com os parâmetros e os resultados de validação cruzada. A tabela abaixo confirma quais registros foram concluídos. A avaliação no teste e o registro do pipeline final são feitos nas seções 14 e 18, respectivamente.

In [0]:
# 12.6 | Conferir o histórico registrado nesta execução
resumo_mlflow = pd.DataFrame(REGISTROS_MLFLOW)
if resumo_mlflow.empty:
    print('Nenhum registro iniciado no MLflow nesta execução.')
else:
    display(resumo_mlflow[['Execução', 'Situação', 'run_id']])
    print(f'Registrados: {(resumo_mlflow["Situação"] == "Registrado").sum()} de {len(resumo_mlflow)} experimentos tentados.')

**Leitura:** a AUC sobe de 0,834 (freio mais forte, `C=0,01`) até 0,838 em `C=1` e para de mudar a partir daí (0,8383 em `C=3` e `C=10`). O ganho de qualquer valor sobre o padrão é de 0,0001, contra uma variação natural de 0,019. **Mantemos `C=1`.** O resultado mostra que o modelo não depende de um ajuste fino: com 18 variáveis e ~4.500 clientes, a regularização padrão já é adequada. Esta etapa documenta que o parâmetro foi testado, e não simplesmente aceito.


## 13. O corte: quando o score vira `Yes`

O desafio pede `Yes`/`No`, então precisamos de um limite de probabilidade. O limite padrão de 0,50 classifica como `Yes` somente probabilidades de pelo menos 50%. Sua escolha não considera automaticamente a capacidade operacional nem os custos de retenção, que não foram informados no desafio.

Comparamos quatro opções usando **apenas as notas "de fora" do treino** (validação cruzada). O conjunto de teste permanece reservado para a avaliação final.

| Opção | Lógica |
|---|---|
| 0,50 | Padrão do software; ignora a proporção real de churn |
| **Taxa média de churn (26,7%)** | `Yes` = cliente com risco acima da média da carteira |
| Ponto de máximo KS | Probabilidade em que o modelo mais separa quem cancela de quem fica |
| Top 20% do ranking | Corte por capacidade: só cabe ligar para 20% da base |


**Nota sobre as bases:** a *validação cruzada* usa somente os 80% de desenvolvimento, divididos em cinco partes; o **teste final** são os 20% separados desde o início. Por isso, KS de validação e KS de teste podem apresentar valores diferentes.


In [0]:
# 13.1 | Comparar opções de corte nas notas de validação cruzada
modelo_escolhido = 'Regressão logística (baseline)'
pipe_final = clone(pipelines[modelo_escolhido]).set_params(modelo__C=C_escolhido)
score_validacao = cross_val_predict(pipe_final, X_bruto_treino, y_treino, cv=cv, method='predict_proba')[:, 1]

fpr, tpr, limiares = roc_curve(y_treino, score_validacao)
corte_ks = float(limiares[np.argmax(tpr - fpr)])
taxa_media = float(y_treino.mean())
corte_top20 = float(np.quantile(score_validacao, 0.80))

def resumo_corte(nome, corte, y_real, score):
    prev = (score >= corte).astype(int)
    return {'Opção': nome, 'Corte': round(corte, 3), '% marcados Yes': round(100*prev.mean(), 1),
            'Precisão': round(precision_score(y_real, prev, zero_division=0), 3), 'Recall': round(recall_score(y_real, prev, zero_division=0), 3),
            'F1': round(f1_score(y_real, prev, zero_division=0), 3)}

df_cortes = pd.DataFrame([resumo_corte('0,50 (padrão)', 0.50, y_treino, score_validacao),
                          resumo_corte('Taxa média de churn', taxa_media, y_treino, score_validacao),
                          resumo_corte('Máximo KS', corte_ks, y_treino, score_validacao),
                          resumo_corte('Top 20% (capacidade)', corte_top20, y_treino, score_validacao)])
display(df_cortes)

# Curva do KS: % acumulado de quem cancelou x de quem ficou, por faixa de score
grade = np.linspace(0, 1, 201)
acum_churn = [(score_validacao[y_treino == 1] <= g).mean() for g in grade]
acum_fica = [(score_validacao[y_treino == 0] <= g).mean() for g in grade]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grade, acum_fica, label='Ficaram (acumulado)', color='#2F6FB5')
ax.plot(grade, acum_churn, label='Cancelaram (acumulado)', color='#E07A2E')
ax.axvline(corte_ks, ls='--', color='grey', label=f'Score no maior KS: {corte_ks:.3f}')
ax.axvline(taxa_media, ls=':', color='black', label=f'Corte adotado: {taxa_media:.3f}')
ax.text(corte_ks + 0.02, 0.08, f'Score de separação = {corte_ks:.3f}', fontsize=9)
ax.text(0.98, 0.10, f'Churn observado no treino = {100*taxa_media:.1f}%', ha='right', fontsize=9)
ax.set_xlabel('Probabilidade estimada de churn'); ax.set_ylabel('% acumulado de clientes'); ax.legend()
ax.set_title(f'KS na validação cruzada (80% de desenvolvimento): {np.max(tpr - fpr):.3f}'); plt.tight_layout(); plt.show()

KS_VALIDACAO = float(np.max(tpr-fpr))
print(f'KS da validação cruzada = {KS_VALIDACAO:.3f}; score em que ocorre = {corte_ks:.3f}; taxa real de churn do treino = {taxa_media:.3f}')

print('Leitura: o KS é a maior diferença vertical entre as duas curvas. O score indicado é o ponto em que essa distância ocorre, não o valor do KS. O KS do teste é calculado separadamente na seção 14.')


### 13.2. Decisão do corte

| Opção | Corte | % marcados Yes | Precisão | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| 0,50 (padrão) | 0,500 | 22% | 65% | 54% | 0,59 |
| **Taxa média de churn** | **0,267** | **41%** | **51%** | **79%** | **0,62** |
| Máximo KS | 0,292 | 39% | 53% | 78% | 0,63 |
| Top 20% (capacidade) | 0,527 | 20% | 66% | 49% | 0,56 |

O corte de máximo KS (0,292) e a taxa média de churn (0,267) praticamente coincidem e dão o mesmo resultado. Com 0,50, quase metade dos clientes que cancelam (46%) passaria como `No`.

**Limitação:** sem estimativas de custo de contato, valor do cliente e efetividade das ações, o corte não representa uma decisão financeira ótima. A escolha foi feita na validação, combinando risco acima da média e proximidade com o ponto de maior KS. O limite poderá ser revisto quando houver informações sobre custos e resultados das ações.

**Corte adotado: a taxa média de churn da base de desenvolvimento.** A justificativa tem três apoios:

1. **Lógica de negócio:** `Yes` = "este cliente tem risco acima da média da carteira". Qualquer pessoa do negócio entende.
2. **Evidência estatística:** o ponto de máximo KS, onde o modelo melhor separa os dois grupos, ficou praticamente no mesmo lugar. Duas lógicas independentes apontam para o mesmo corte.
3. **Funciona porque a logística é calibrada:** a probabilidade que ela gera corresponde à taxa real observada, então "acima de 26,7%" significa de fato "acima da média".

**O que se ganha e o que se paga:** em relação ao 0,50, o recall sobe muito (deixamos escapar bem menos cancelamentos), e a precisão cai (mais ligações para quem não ia sair). A decisão operacional deverá considerar o custo de contato e a capacidade de atendimento.

**Para o dia a dia, faixas de risco** (todas derivadas do treino):

| Faixa | Regra | Ação sugerida |
|---|---|---|
| **Alto** | top 20% de score | Ligação ativa com oferta de retenção |
| **Médio** | acima da média, fora do top 20% | Contato de menor custo (e-mail, SMS, oferta de fidelidade) |
| **Baixo** | abaixo da média → `No` | Sem ação |


**Leitura do gráfico de KS:** o valor de **KS** mede o afastamento máximo entre as curvas de clientes que cancelaram e que permaneceram. O número **0,29 é o score em que essa distância ocorre**, não o KS. A taxa de 0,267 é o percentual de cancelamentos no treino, também não é o KS.


## 14. Avaliação final no conjunto de teste

Após a definição das variáveis, do modelo e do corte na etapa de desenvolvimento, a regressão logística é ajustada com os dados de treino e avaliada nos **1.127 clientes do conjunto de teste reservado**.


O gráfico ROC desta seção mostra **AUC e KS do teste de 20%**, enquanto o gráfico anterior mostra **KS da validação cruzada dentro dos 80% de desenvolvimento**. São amostras e avaliações diferentes.


In [0]:
# 14.1 | Treinar no treino inteiro e avaliar no teste
modelo_validado = clone(pipe_final).fit(X_bruto_treino, y_treino)
score_teste = modelo_validado.predict_proba(X_bruto_teste)[:, 1]
CORTE = taxa_media
prev_teste = (score_teste >= CORTE).astype(int)

fpr_t, tpr_t, _ = roc_curve(y_teste, score_teste)
auc_teste = roc_auc_score(y_teste, score_teste)
metricas_teste = pd.DataFrame([{
    'AUC': auc_teste, 'Gini': 2*auc_teste - 1, 'KS': np.max(tpr_t - fpr_t),
    'Precisão': precision_score(y_teste, prev_teste, zero_division=0), 'Recall': recall_score(y_teste, prev_teste, zero_division=0),
    'F1': f1_score(y_teste, prev_teste, zero_division=0), '% marcados Yes': prev_teste.mean()}]).round(3)
print('Métricas no teste (corte = taxa média de churn):'); display(metricas_teste)

vp = int(((prev_teste == 1) & (y_teste == 1)).sum()); fp = int(((prev_teste == 1) & (y_teste == 0)).sum())
fn = int(((prev_teste == 0) & (y_teste == 1)).sum()); vn = int(((prev_teste == 0) & (y_teste == 0)).sum())
display(pd.DataFrame({'Marcado Yes': [vp, fp], 'Marcado No': [fn, vn]},
                     index=['Cancelou de fato', 'Ficou de fato']))
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(fpr_t, tpr_t, color='#E07A2E', lw=2.4, label=f'Regressão logística | AUC = {auc_teste:.3f}')
ax.plot([0, 1], [0, 1], ls='--', color='grey', label='Sorteio | AUC = 0,5')
i_ks = np.argmax(tpr_t - fpr_t)
ax.vlines(fpr_t[i_ks], fpr_t[i_ks], tpr_t[i_ks], color='black', lw=1.2)
ax.text(fpr_t[i_ks] + 0.02, (fpr_t[i_ks] + tpr_t[i_ks]) / 2, f'KS = {tpr_t[i_ks]-fpr_t[i_ks]:.2f}')
ax.set_xlabel('% de quem ficou marcado como Yes (alarme falso)'); ax.set_ylabel('% de quem cancelou marcado como Yes (recall)')
ax.set_title(f'Curva ROC — 20% de teste | AUC={auc_teste:.3f} | KS={np.max(tpr_t-fpr_t):.3f}'); ax.legend(loc='lower right'); plt.tight_layout(); plt.show()
print(f'Leitura: dos {vp+fn} clientes de teste que cancelaram, o modelo marcou {vp} ({100*vp/(vp+fn):.0f}%). '
      f'Dos {vp+fp} marcados Yes, {vp} cancelaram de fato ({100*vp/(vp+fp):.0f}%).')

registrar_resultado_mlflow('avaliacao_final_reglog',
    {'auc_teste': auc_teste, 'ks_teste': float(np.max(tpr_t-fpr_t))},
    {'algoritmo': modelo_escolhido, 'C': float(C_escolhido), 'corte': float(CORTE),
     'base': 'teste_20pct', 'pipeline': 'tratamentos_e_modelo'})


**Matriz de confusão:** as linhas mostram o que de fato aconteceu; as colunas, a previsão do modelo. A diagonal traz os acertos. A célula “Cancelou/No” indica cancelamentos não identificados; “Ficou/Yes”, clientes sinalizados que não cancelaram.

**Curva ROC:** mostra a capacidade de separar quem cancelou de quem permaneceu em diversos limites de score. **AUC** resume a curva inteira; o **KS do teste** é a maior separação entre as taxas acumuladas dos dois grupos. O valor pode diferir do KS da validação cruzada, porque as bases são diferentes.

### 14.2. Taxa de cancelamento por decil

Os clientes do teste são separados em dez grupos de tamanhos semelhantes: **decil 1 = menor score**, **decil 10 = maior score**. O objetivo é verificar se a taxa real de cancelamento aumenta junto com o risco previsto.


In [0]:
# 14.2 | Decis: grupos de 10% da base, score crescente e captura acumulada
ranking = pd.DataFrame({'churn_real': y_teste.to_numpy(), 'score': score_teste}).sort_values('score').reset_index(drop=True)
ranking['decil'] = np.floor(np.arange(len(ranking)) * 10 / len(ranking)).astype(int) + 1
media_teste = float(ranking['churn_real'].mean())
decis = ranking.groupby('decil').agg(clientes=('churn_real', 'size'), cancelamentos=('churn_real', 'sum'),
    taxa_churn_real=('churn_real', 'mean'), score_min=('score', 'min'), score_medio=('score', 'mean'), score_max=('score', 'max'))
decis['pct_clientes'] = 100*decis['clientes']/len(ranking)
decis['lift'] = decis['taxa_churn_real']/media_teste

resumo_decis = pd.DataFrame({
    'Decil': ['D'+str(x) for x in decis.index], 'Clientes': decis['clientes'].to_numpy(),
    '% clientes': decis['pct_clientes'].to_numpy(), 'Score mínimo': decis['score_min'].to_numpy(),
    'Score médio': decis['score_medio'].to_numpy(), 'Score máximo': decis['score_max'].to_numpy(),
    'Churn real (%)': 100*decis['taxa_churn_real'].to_numpy(), 'Lift': decis['lift'].to_numpy()
})
print('D1 = os 10% de menor score; D10 = os 10% de maior score. Scores na tabela são probabilidades estimadas, não limites fixos.')
display(resumo_decis.round(3))

fig, ax = plt.subplots(figsize=(11.5, 5.2))
idx = decis.index.to_numpy(); ax2 = ax.twinx()
barras = ax.bar(idx, decis['pct_clientes'], color='#CAD8E9', width=.72, label='% dos clientes no decil')
linha, = ax2.plot(idx, 100*decis['taxa_churn_real'], color='#E07A2E', marker='o', lw=2.5, label='Cancelamento real (%)')
media = ax2.axhline(100*media_teste, ls='--', color='#444444', lw=1.1, label=f'Média no teste: {100*media_teste:.1f}%')
for d, taxa in zip(idx, decis['taxa_churn_real']):
    ax2.annotate(f'{100*taxa:.0f}%', (d, 100*taxa), xytext=(0, 7), textcoords='offset points', ha='center', color='#B35C13', fontsize=9, fontweight='bold')
ax.set_xticks(idx, [f'D{d}\n{decis.loc[d, "score_medio"]:.2f}'.replace('.', ',') for d in idx])
ax.set_ylabel('% de clientes por decil (barras)'); ax2.set_ylabel('Taxa real de cancelamento (%) (linha)')
ax.set_ylim(0, 16); ax2.set_ylim(0, 105)
ax.set_xlabel('Decil (1ª linha) e score médio do grupo (2ª linha) | menor risco → maior risco')
ax.set_title('Decis de score: grupos de 10% e cancelamentos observados')
ax2.legend([barras, linha, media], ['% dos clientes (10% por grupo)', 'Churn real por decil', f'Média no teste: {100*media_teste:.1f}%'], loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()
print(f'Leitura dos decis: D1 reúne os 10% de menor score (média {decis.loc[1,"score_medio"]:.2f}) e teve {100*decis.loc[1,"taxa_churn_real"]:.1f}% de cancelamentos. D10 reúne os 10% de maior score (média {decis.loc[10,"score_medio"]:.2f}) e teve {100*decis.loc[10,"taxa_churn_real"]:.1f}%. A comparação mostra se o risco estimado acompanha o cancelamento realmente observado.')

# Captura é calculada do MAIOR score para o MENOR, mesmo com os decis do gráfico acima em ordem crescente.
decis_captura = decis.sort_index(ascending=False).copy()
decis_captura['captura_acumulada'] = decis_captura['cancelamentos'].cumsum()/decis_captura['cancelamentos'].sum()
x_captura = np.r_[0, 100*decis_captura['clientes'].cumsum().to_numpy()/len(ranking)]
y_captura = np.r_[0, 100*decis_captura['captura_acumulada'].to_numpy()]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x_captura, y_captura, marker='o', lw=2.4, color='#E07A2E', label='Modelo: priorizar maior score')
ax.plot([0, 100], [0, 100], '--', color='#7B7B7B', label='Sem priorização (aleatório)')
ax.set_xlim(0, 100); ax.set_ylim(0, 105); ax.grid(alpha=.18)
ax.set_xlabel('% de clientes abordados, começando pelo maior risco'); ax.set_ylabel('% dos cancelamentos identificados')
ax.set_title('Captura acumulada: quanto churn está concentrado nos maiores scores?')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()
clientes_top20 = int(decis_captura['clientes'].iloc[:2].sum())
churn_top20 = int(decis_captura['cancelamentos'].iloc[:2].sum())
pct_churn_top20 = 100*churn_top20/int(ranking['churn_real'].sum())
print(f'Leitura da captura: os {clientes_top20} clientes dos dois decis com maior score representam aproximadamente 20% do teste e concentram {churn_top20} dos {int(ranking["churn_real"].sum())} cancelamentos ({pct_churn_top20:.1f}%). A diagonal representa uma escolha sem priorização: em 20% dos clientes, esperaríamos encontrar cerca de 20% dos cancelamentos.')

**Decis:** cada barra representa cerca de 10% dos clientes do teste. O **decil 1 (D1)** reúne os menores scores e o **decil 10 (D10)**, os maiores. A linha mostra a taxa real de cancelamento. A segunda linha do eixo horizontal informa o score médio de cada decil; a tabela traz também os scores mínimo e máximo. Assim, é possível conferir se o risco previsto cresce junto com a proporção de cancelamentos observados.

**Captura acumulada:** para priorizar abordagens, começamos pelos clientes com **maior score** (D10, depois D9...). O gráfico informa que parcela dos cancelamentos observados é encontrada à medida que aumentamos a quantidade de clientes considerados. A diagonal representa uma escolha sem priorização. O percentual efetivo dos dois decis de maior score é exibido abaixo do gráfico.

### 14.4. Calibração: o score corresponde ao risco real?

**Por que esta etapa:** o corte da seção 13 diz "`Yes` = probabilidade acima da média da carteira". Isso só vale se a probabilidade do modelo for **calibrada**: entre clientes com score 0,30, cerca de 30% devem cancelar de fato. Um modelo pode ordenar bem (AUC alta) e ainda assim dar probabilidades exageradas ou tímidas.

**Como medimos:** nos decis do teste, comparamos o **score médio** (o que o modelo previu) com a **taxa real de churn** (o que aconteceu). Se os pontos caem sobre a diagonal, o modelo está calibrado. Complementamos com o **Brier score**: a média de (probabilidade − resultado real)², em que menor é melhor e o valor de referência é o de "prever a média para todo mundo".


**Calibração perfeita:** é a linha diagonal em que uma probabilidade prevista de 30% corresponde a 30% de cancelamentos observados. Os pontos da regressão logística mostram, para cada decil, o **score médio previsto** no eixo horizontal e a **taxa real de cancelamento** no vertical. Quanto mais próximos da diagonal, melhor a calibração.




In [0]:
# 14.4 | Calibração no teste
from sklearn.metrics import brier_score_loss
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], ls='--', color='grey', label='Calibração perfeita')
ax.plot(decis['score_medio'], decis['taxa_churn_real'], marker='o', color='#E07A2E', label='Regressão logística (decis)')
ax.set_xlabel('Score médio do decil (previsto)'); ax.set_ylabel('Taxa real de churn do decil (observado)')
ax.set_title('Calibração no teste'); ax.legend(); plt.tight_layout(); plt.show()
brier = brier_score_loss(y_teste, score_teste); brier_ref = brier_score_loss(y_teste, np.full(len(y_teste), y_treino.mean()))
display(pd.DataFrame({'Score médio previsto': [score_teste.mean()], 'Taxa real de churn': [y_teste.mean()],
                      'Maior diferença entre decis': [(decis['score_medio'] - decis['taxa_churn_real']).abs().max()],
                      'Brier do modelo': [brier], 'Brier de referência (prever a média)': [brier_ref]}).round(4))


**Interpretação da calibração:** cada ponto compara o **score médio previsto** de um decil com sua **taxa real de cancelamento**. A diagonal pontilhada representa a calibração perfeita: por exemplo, um grupo com score médio de 30% apresentaria aproximadamente 30% de cancelamentos. Quanto mais próximos os pontos da diagonal, melhor a correspondência entre risco estimado e risco observado.

**Brier:** mede o erro das probabilidades previstas frente ao resultado real (cancelou = 1; permaneceu = 0). Quanto **menor**, melhor; a tabela compara o modelo com uma previsão simples baseada na taxa média de cancelamento.

## 15. Quais variáveis pesam no risco

Duas visões complementares:

- **Importância por permutação** (*permutation importance*): após treinar e escolher o modelo, embaralhamos uma variável por vez na base de teste, mantendo as demais. Medimos quanto a AUC diminui. Quanto maior a queda, mais aquela informação contribui para a capacidade de ordenar clientes pelo risco. O valor é uma **queda de AUC**, não um percentual de contribuição. Essa análise explica o modelo escolhido; não altera a seleção das variáveis.
- **Pesos da regressão logística em razão de chances** (*odds ratio*): diz para que lado e quanto cada característica move o risco, **mantidas as demais constantes**. Razão de chances 2 = dobra a chance de cancelar; 0,5 = corta pela metade. Categorias são comparadas à de referência: contrato mensal, DSL, transferência bancária, etc.


In [0]:
# 15.1 | Importância por permutação sobre as variáveis da ABT já tratada
# Embaralhar dados originais antes da imputação cria combinações impossíveis de ausência.
# Por isso, a permutação é feita após o tratamento, preservando o conjunto de variáveis selecionadas.
modelo_importancia = Pipeline([
    ('preparacao', clone(preprocessamento)),
    ('modelo', clone(pipe_final.named_steps['modelo']))
]).fit(X_treino, y_treino)
assert np.allclose(modelo_importancia.predict_proba(X_teste)[:, 1], score_teste, atol=1e-8), 'Modelos divergentes na análise de importância.'
perm = permutation_importance(modelo_importancia, X_teste, y_teste, scoring='roc_auc', n_repeats=10, random_state=42)
importancia = pd.Series(perm.importances_mean, index=X_teste.columns).sort_values(ascending=False)
top = importancia.head(10)[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(top.index, top.values, color='#2F6FB5'); ax.set_xlabel('Redução da AUC ao embaralhar a variável')
ax.set_title('Importância das variáveis selecionadas da ABT'); plt.tight_layout(); plt.show()
display(importancia.round(4).rename('Queda na AUC').to_frame().head(10))
print('Interpretação: maior queda de AUC indica maior dependência do modelo da informação dessa variável. É importância preditiva, não efeito causal.')

In [0]:
# 15.2 | Pesos da regressão logística em razão de chances
nomes = modelo_validado.named_steps['preparacao'].get_feature_names_out()
coef = modelo_validado.named_steps['modelo'].coef_[0]
pesos = pd.DataFrame({'Variável': [n.replace('cat__', '').replace('num__', '') for n in nomes],
                      'Coeficiente': coef, 'Razão de chances': np.exp(coef)})
pesos['Efeito'] = np.where(pesos['Coeficiente'] > 0, 'aumenta o risco', 'reduz o risco')
pesos = pesos.reindex(pesos['Coeficiente'].abs().sort_values(ascending=False).index)
print('Numéricas: efeito de +1 desvio-padrão. Categóricas: efeito vs categoria de referência.')
display(pesos.head(12).round(3).reset_index(drop=True))


### 15.3. Leitura

**O que mais pesa (permutação):** o **tempo de relacionamento** domina com folga — embaralhá-lo derruba a AUC em 0,14. Em seguida vêm **tipo de internet** (0,05), **tipo de contrato** (0,03), **total pago** e **mensalidade** (0,01 cada). As demais têm peso pequeno individualmente.

**Para que lado (razão de chances, mantidas as demais constantes):**

- **Contrato bienal** tem **1/4 da chance** de cancelar de quem tem contrato mensal (0,25); o anual, metade (0,49).
- **Mais tempo de casa** reduz a chance a **1/3** a cada aumento de um desvio-padrão (~24 meses) (0,33).
- **Fibra** quase **triplica** a chance em relação ao DSL (2,89).
- **Suporte técnico** e **segurança online** reduzem a chance em cerca de 30% (0,70 e 0,74).
- **Cheque eletrônico** e **fatura digital** aumentam cerca de 30% (1,33 e 1,34).

Todos esses sentidos batem com a análise exploratória da seção 05: o modelo aprendeu padrões que fazem sentido de negócio.

**Dois pesos que exigem cuidado:**
- **Mensalidade aparece reduzindo o risco (0,69)**, embora na exploratória quem cancelou pagasse mais. O motivo: o preço já está representado pelos serviços que o compõem (fibra, streaming) e pelo total pago. "Mantidas as demais constantes" significa comparar clientes com os mesmos serviços, e aí a mensalidade que sobra tem outro significado. É efeito de variáveis que andam juntas, não um achado de negócio.
- **Streaming de TV e de filmes aparecem aumentando o risco (1,57 e 1,38)** pela mesma razão: refletem pacote e preço, não o streaming em si.

**Interpretação:** a importância por permutação mostra a contribuição da variável para a discriminação do modelo. As razões de chances descrevem associações condicionadas às demais preditoras e requerem cuidado quando há variáveis relacionadas.

**Limitação:** tempo de relacionamento e total pago apresentam associação entre si. Nessas situações, os coeficientes isolados da regressão podem ser menos estáveis e não devem ser interpretados como efeitos causais.


### 15.4. As variáveis criadas pelo grupo ajudam?

Na ABT (seção 08) o grupo criou três variáveis e, na seção 10, o indicador de total ausente:

| Variável | Pergunta de negócio |
|---|---|
| `comprometimentoRenda` | A mensalidade pesa no orçamento da família? |
| `avgMonthlyCharges` | Quanto o cliente pagou, em média, por mês ao longo do contrato? |
| `evolucaoMonthlyCharges` | A mensalidade atual está acima ou abaixo da média histórica registrada? |
| `TotalCharges_ausente` | O total pago veio vazio no cadastro? |

Testamos de duas formas: (1) a taxa de churn muda entre faixas da variável? (2) o modelo fica melhor com elas do que sem elas?


In [0]:
# 15.4 | Avaliar as variáveis criadas, incluindo faixas com valores repetidos
criadas = ['comprometimentoRenda', 'avgMonthlyCharges', 'evolucaoMonthlyCharges']
base_c = X_treino_completo.assign(churn=y_treino.to_numpy())
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, v in zip(axes, criadas):
    valores = pd.to_numeric(base_c[v], errors='coerce')
    if valores.dropna().nunique() < 2:
        ax.text(.5, .5, 'Sem variação suficiente para formar faixas', ha='center', va='center', transform=ax.transAxes, fontsize=9)
        ax.set_title(v); ax.set_axis_off(); continue
    faixa = pd.qcut(valores, q=5, duplicates='drop')
    taxa = base_c.groupby(faixa, observed=True)['churn'].mean().mul(100)
    x = np.arange(1, len(taxa)+1)
    ax.bar(x, taxa.values, color='#7FA6D6')
    ax.axhline(100*y_treino.mean(), ls='--', color='black', lw=1)
    for pos, t in zip(x, taxa.values): ax.text(pos, t + 1, f'{t:.0f}%', ha='center', fontsize=9)
    ax.set_xticks(x, [f'F{j}' for j in x]); ax.set_title(v)
    ax.set_xlabel(f'Faixas de valores ({len(taxa)} grupos)'); ax.set_ylabel('Churn (%)')
plt.tight_layout(); plt.show()
print('Leitura: as faixas comparam a taxa de cancelamento em diferentes valores das variáveis criadas. Quando há muitos valores repetidos, o número de faixas pode ser menor que cinco para evitar divisões artificiais.')
print('Taxa de cancelamento para registros cujo TotalCharges original estava ausente:')
display((base_c.groupby('TotalCharges_ausente')['churn'].agg(['size', 'mean']) * [1, 100]).round(1)
        .rename(columns={'size': 'Clientes', 'mean': 'Churn (%)'}))

# Mesmo modelo, com e sem as variáveis criadas; comparação no conjunto de desenvolvimento.
def auc_cv(colunas):
    base = X_treino_completo
    cat = [c for c in colunas if not pd.api.types.is_numeric_dtype(base[c])]
    num = [c for c in colunas if c not in cat]
    prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), cat),
                              ('num', StandardScaler(), num)])
    return cross_validate(Pipeline([('p', prep), ('m', LogisticRegression(max_iter=3000))]),
                          base[colunas], y_treino, cv=cv, scoring='roc_auc')['test_score'].mean()
todas = list(X_treino_completo.columns)
sem_criadas = [c for c in todas if c not in criadas + ['TotalCharges_ausente']]
display(pd.DataFrame({'Versão': ['Só variáveis originais', 'Originais + criadas'],
                      'AUC validação': [auc_cv(sem_criadas), auc_cv(todas)]}).round(4))

**Leitura:** as variáveis criadas **não acrescentaram poder ao modelo**: a AUC é 0,840 com e sem elas. O motivo é que elas repetem informação que o modelo já tem.

- `avgMonthlyCharges` mostra churn subindo de 10% para 33–37% entre as faixas, mas é praticamente a própria mensalidade (`MonthlyCharges`), que já está no modelo.
- `comprometimentoRenda` separa só a faixa de menor comprometimento (14%); as demais ficam em torno de 30%. A renda sozinha quase não diferencia (seção 05), e a mensalidade já carrega o resto.
- `evolucaoMonthlyCharges` tem pico de 42% na faixa do meio. Investigando: nessa faixa a razão é ≈ 1, e o grupo com razão exatamente 1 (13% dos clientes) tem **mediana de 1 mês de casa** e 50% de churn. Ou seja, ela está captando **cliente novo**, que o `tenure` já capta — não reajuste de preço.
- `TotalCharges_ausente`: 32% de churn contra 26%, diferença pequena e em poucos clientes (215 no treino). Por isso saiu na seleção por IV (seção 10.2).

**Decisão:** as três variáveis de valor passaram no corte de IV (todas acima de 0,15). A **mensalidade média** saiu depois, na etapa de correlação (seção 10.3), por ser cópia da mensalidade atual (0,997). **Evolução da mensalidade** e **comprometimento de renda** continuam no modelo. Mantê-las não prejudica (a AUC não cai, o PSI delas é estável) e preserva a leitura de negócio proposta pelo grupo. Uma análise mais precisa da evolução da mensalidade exigiria o **histórico real de faturas**, que não está disponível nesta base.


### 15.5. Conferência de consistência do pipeline

O pipeline definido na seção 10.4 é utilizado na validação e no teste. Esta conferência compara suas probabilidades com as de uma execução equivalente sobre a ABT previamente preparada, verificando se as transformações e o modelo apresentam resultados consistentes.


In [0]:
# 15.5 | Conferir consistência entre pipeline e ABT preparada
# Mesma regressão logística e mesmas 18 variáveis; uma execução recebe a ABT preparada.
modelo_analitico = Pipeline([
    ('preparacao', clone(preprocessamento)),
    ('modelo', clone(pipe_final.named_steps['modelo']))
]).fit(X_treino, y_treino)
score_analitico = modelo_analitico.predict_proba(X_teste)[:, 1]
score_pipeline = modelo_validado.predict_proba(X_bruto_teste)[:, 1]
diferenca_maxima = float(np.max(np.abs(score_analitico - score_pipeline)))
display(pd.DataFrame({
    'Implementação':['ABT preparada','Pipeline integral'],
    'AUC no teste':[roc_auc_score(y_teste,score_analitico), roc_auc_score(y_teste,score_pipeline)]
}).round(4))
print(f'Maior diferença entre as probabilidades: {diferenca_maxima:.10f}')
assert np.allclose(score_analitico,score_pipeline,atol=1e-8), 'Tratamentos do pipeline divergiram da ABT preparada.'
print('Conferência aprovada: mesmo modelo e mesmas probabilidades.')


**Leitura:** uma diferença máxima próxima de zero confirma que a preparação dentro do pipeline reproduz a ABT original. Portanto, a mesma transformação pode ser aplicada a novos clientes com uma única sequência de transformações para desenvolvimento e inferência.


## 16. Previsão para a base de inferência

Com modelo e corte definidos, treinamos a regressão logística com **todo o histórico** (treino + teste: mais dados, mesmas regras) e pontuamos os 1.409 clientes. O corte e as faixas são os definidos no treino, sem recalcular.

O arquivo de entrega tem `customerID` e `prediction`. As probabilidades e faixas de risco ficam disponíveis em uma tabela auxiliar para apoiar a priorização das ações de retenção.


In [0]:
# 16.1 | Treinar o pipeline completo e pontuar a inferência
import datetime

# Depois da avaliação independente, reajustamos o MESMO pipeline com todo o histórico.
modelo_final = clone(pipeline_base).fit(X_bruto_historico, y)
prob_inferencia = modelo_final.predict_proba(X_bruto_inferencia)[:, 1]

resultado_inferencia = pd.DataFrame({
    'customerID': df_inferencia['customerID'].to_numpy(),
    'probabilidade_churn': prob_inferencia
})
resultado_inferencia['prediction'] = np.where(prob_inferencia >= CORTE, 'Yes', 'No')
resultado_inferencia['faixa_risco'] = np.select(
    [prob_inferencia >= corte_top20, prob_inferencia >= CORTE],
    ['Alto', 'Médio'], default='Baixo'
)

# Arquivo obrigatório: manter a ordem e os IDs originais; ranking fica separado.
submissao = resultado_inferencia[['customerID', 'prediction']].copy()
ranking_inferencia = resultado_inferencia.sort_values(
    'probabilidade_churn', ascending=False
).reset_index(drop=True)
assert len(submissao) == 1409 and submissao['customerID'].is_unique
assert submissao['customerID'].tolist() == df_inferencia['customerID'].tolist()
assert set(submissao['prediction']).issubset({'Yes', 'No'})

nome_csv = f'prediction_Grupo6_{datetime.datetime.now():%Y%m%d_%H%M%S}.csv'
submissao.to_csv(nome_csv, index=False)
print(f'CSV gerado: {nome_csv} | Clientes: {len(submissao)} | Corte Yes/No: {CORTE:.4f}')
display(resultado_inferencia['faixa_risco'].value_counts()
        .reindex(['Alto', 'Médio', 'Baixo']).rename('Clientes').to_frame())
display(ranking_inferencia.head(10).round(3))


## 17. Estabilidade: a base nova se parece com a de desenvolvimento?

Na inferência não sabemos quem vai cancelar, então não dá para calcular AUC ou KS. O que dá para checar é se **os clientes novos se parecem com os que o modelo conheceu**. Se a base mudou muito, as notas do modelo merecem desconfiança.

O **PSI** (*Population Stability Index*, índice de estabilidade populacional) faz essa checagem:

1. Cortamos o score em 10 faixas fixas (0,0–0,1; 0,1–0,2; ...; 0,9–1,0).
2. Calculamos o % de clientes em cada faixa no **histórico completo** (referência) e na **inferência**, usando o **mesmo pipeline final treinado com todo o histórico** para pontuar ambas as bases.
3. Somamos, faixa a faixa: `(% inferência − % referência) × ln(% inferência / % referência)`.

| PSI | Leitura de mercado |
|---|---|
| abaixo de 0,10 | estável — pode usar |
| 0,10 a 0,25 | atenção — investigar o que mudou |
| acima de 0,25 | mudança relevante — revisar o modelo |

Fazemos o mesmo para cada variável, para saber **onde** estaria uma eventual mudança.


In [0]:
# 17.1 | PSI do score e das variáveis: histórico (referência) x inferência
PSI_FAIXAS = np.linspace(0, 1, 11)
def psi_de_proporcoes(p, q):
    p = np.maximum(np.asarray(p, float), 1e-6); q = np.maximum(np.asarray(q, float), 1e-6)
    return float(np.sum((q - p) * np.log(q / p)))
def psi_numerico(ref, atual, cortes=None):
    ref = np.asarray(ref, float); atual = np.asarray(atual, float)
    c = np.unique(np.quantile(ref, np.linspace(0, 1, 11))) if cortes is None else np.array(cortes, float)
    if len(c) < 3:
        return psi_categorico(ref, atual)
    c[0], c[-1] = -np.inf, np.inf
    return psi_de_proporcoes(np.histogram(ref, c)[0] / len(ref), np.histogram(atual, c)[0] / len(atual))
def psi_categorico(ref, atual):
    a = pd.Series(ref).astype(str).value_counts(normalize=True); b = pd.Series(atual).astype(str).value_counts(normalize=True)
    cats = a.index.union(b.index)
    return psi_de_proporcoes(a.reindex(cats, fill_value=0), b.reindex(cats, fill_value=0))
def leitura_psi(v): return 'estável' if v < 0.10 else 'atenção' if v <= 0.25 else 'mudança relevante'

score_ref = modelo_final.predict_proba(X_bruto_historico)[:, 1]
score_inf = prob_inferencia
psi_score = psi_numerico(score_ref, score_inf, PSI_FAIXAS)

c = PSI_FAIXAS.copy(); c[0], c[-1] = -np.inf, np.inf
pct_ref = 100 * np.histogram(score_ref, c)[0] / len(score_ref)
pct_inf = 100 * np.histogram(score_inf, c)[0] / len(score_inf)
rotulos = [f'{PSI_FAIXAS[i]:.1f}–{PSI_FAIXAS[i+1]:.1f}'.replace('.', ',') for i in range(10)]
pos = np.arange(10)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(pos - 0.2, pct_ref, width=0.4, color='#2F6FB5', label='Histórico de desenvolvimento (referência)')
ax.bar(pos + 0.2, pct_inf, width=0.4, color='#E07A2E', label='Inferência (clientes novos)')
ax.axvline(CORTE * 10 - 0.5, ls='--', color='black', lw=1.2, label=f'Corte Yes/No = {CORTE:.3f}'.replace('.', ','))
ax.set_xticks(pos, rotulos, rotation=30)
ax.set_xlabel('Faixa de score (maior risco à direita)'); ax.set_ylabel('% dos clientes da base')
ax.set_title(f'Distribuição do score: PSI = {psi_score:.3f} ({leitura_psi(psi_score)})'.replace('.', ',', 1))
ax.legend(); plt.tight_layout(); plt.show()

psi_vars = []
for v in X.columns:
    cat = v in cols_cat or X[v].nunique() <= 2
    p = psi_categorico(X[v], X_inferencia[v]) if cat else psi_numerico(X[v], X_inferencia[v])
    psi_vars.append({'Variável': v, 'PSI': p, 'Leitura': leitura_psi(p)})
psi_vars = pd.DataFrame(psi_vars).sort_values('PSI', ascending=False).reset_index(drop=True)
print(f'PSI do score = {psi_score:.4f} → {leitura_psi(psi_score)}')
print(f'% acima do corte: {100*(score_ref >= CORTE).mean():.1f}% no histórico e {100*(score_inf >= CORTE).mean():.1f}% na inferência')
print(f'Maior PSI entre as variáveis: {psi_vars.iloc[0, 0]} = {psi_vars.iloc[0, 1]:.4f}')
display(psi_vars.head(10).round(4))


**Como ler o gráfico:** cada par de barras é uma faixa de score. Barras azul e laranja da mesma altura = a proporção de clientes naquela faixa é igual nas duas bases. A linha tracejada é o corte: tudo à direita dela vira `Yes`.

**Resultado:** PSI do score = **0,003**, muito abaixo de 0,10 — **estável**. As barras azul e laranja praticamente coincidem em todas as faixas, e a proporção de clientes acima do corte é a mesma nas duas bases (**41,0%** no histórico e **41,4%** na inferência).

Por variável, o maior PSI é o do `tenure`, com aproximadamente **0,010** — também estável. Nenhuma variável muda de perfil entre as bases.

**Conclusão:** não foram observadas mudanças relevantes nas distribuições do score e das variáveis monitoradas entre as bases. O PSI não mede acerto das previsões nem garante que a AUC histórica se repetirá na inferência; isso só poderá ser verificado quando os resultados de cancelamento forem conhecidos.


## 18. Registro do modelo final no MLflow

A seção 12 registra a comparação dos algoritmos e o ajuste da regressão logística. A seção 14 registra as métricas no teste independente. Esta etapa registra o **pipeline completo do modelo final** no mesmo experimento ativo, com tratamentos, variáveis selecionadas, codificação, escala, coeficientes e o corte `Yes/No`.

O identificador da execução e a URI do modelo são exibidos abaixo para permitir sua recuperação e conferência na seção 18.2.


In [0]:
# 18.1 | Registrar pipeline e modelo final no MLflow, com registro tolerante a falhas
URI_MODELO_MLFLOW = None
RUN_MODELO_MLFLOW = None
if not MLFLOW_DISPONIVEL:
    print('Registro no MLflow pendente: biblioteca indisponível. Previsões e CSV continuam válidos.')
else:
    try:
        import mlflow.pyfunc
        from mlflow.models import infer_signature
        class ModeloChurnPublicavel(mlflow.pyfunc.PythonModel):
            def __init__(self, pipeline, corte):
                self.pipeline = pipeline
                self.corte = float(corte)

            def predict(self, context, model_input, params=None):
                score = self.pipeline.predict_proba(model_input)[:, 1]
                return pd.DataFrame({
                    'probabilidade_churn': score,
                    'prediction': np.where(score >= self.corte, 'Yes', 'No')
                })

        exemplo_entrada = X_bruto_inferencia.head(3).copy()
        exemplo_saida = ModeloChurnPublicavel(modelo_final, CORTE).predict(None, exemplo_entrada)
        assinatura = infer_signature(exemplo_entrada, exemplo_saida)

        with mlflow.start_run(experiment_id=MLFLOW_EXPERIMENTO_ID, run_name='terra_signal_reglog_pipeline_final') as run:
            mlflow.log_params({
                'modelo': 'regressao_logistica',
                'regressao_C': float(C_escolhido),
                'corte_yes_no': float(CORTE),
                'quantidade_variaveis': len(COLUNAS_MODELO),
                'quantidade_registros_treino': len(y),
                'validacao': 'estratificada_80_20_cv5',
                'referencia_classe_positiva': 'Churn=Yes',
                'feedback_no_modelo': 'nao'
            })
            mlflow.log_metrics({
                'auc_teste': float(auc_teste),
                'ks_teste': float(metricas_teste['KS'].iloc[0]),
                'brier_teste': float(brier),
                'psi_score_teste_inferencia': float(psi_score)
            })
            mlflow.log_text('\n'.join(COLUNAS_MODELO), 'variaveis_modelo.txt')
            mlflow.log_text(
                'Entrada: colunas originais sem customerID, CustomerFeedback e Churn. '
                'Saída: probabilidade_churn e prediction (Yes/No). '
                'Limiar informado no parâmetro corte_yes_no. '
                'O conjunto de teste não foi utilizado na seleção do modelo.',
                'guia_modelo.txt'
            )
            # MLflow 3 usa 'name'; MLflow 2 usa 'artifact_path'.
            parametros_modelo = dict(
                python_model=ModeloChurnPublicavel(modelo_final, CORTE),
                input_example=exemplo_entrada,
                signature=assinatura
            )
            versao_maior_mlflow = int(mlflow.__version__.split('.')[0])
            if versao_maior_mlflow >= 3:
                parametros_modelo['name'] = 'modelo_churn'
            else:
                parametros_modelo['artifact_path'] = 'modelo_churn'
            informacoes_modelo = mlflow.pyfunc.log_model(**parametros_modelo)
            URI_MODELO_MLFLOW = informacoes_modelo.model_uri

        registro_confirmado = mlflow.get_run(run.info.run_id)
        if str(registro_confirmado.info.experiment_id) != MLFLOW_EXPERIMENTO_ID:
            raise RuntimeError('O modelo foi registrado em um experimento diferente do configurado.')
        print('MLflow registrado | run_id:', run.info.run_id)
        print('URI do modelo:', URI_MODELO_MLFLOW)
        print('O pipeline pode ser recuperado por esta URI para reproduzir as previsões.')
        RUN_MODELO_MLFLOW = run.info.run_id
    except Exception as exc:
        # Evita perder a geração do CSV por problema externo de permissão ou sessão MLflow.
        URI_MODELO_MLFLOW = None
        print(f'Registro no MLflow pendente: {type(exc).__name__}: {exc}')
        print('O modelo já foi treinado e o CSV foi gerado; o registro pode ser reexecutado separadamente.')

In [0]:
# 18.2 | Validar o modelo recuperado, somente quando houve registro bem-sucedido
if URI_MODELO_MLFLOW is None:
    print('Conferência do MLflow não aplicável: não há URI registrada nesta execução.')
else:
    try:
        modelo_recuperado = mlflow.pyfunc.load_model(URI_MODELO_MLFLOW)
        conferencia = modelo_recuperado.predict(X_bruto_inferencia.head(5))
        esperado = pd.DataFrame({'probabilidade_churn': prob_inferencia[:5],
                                 'prediction': submissao['prediction'].iloc[:5].to_numpy()})
        assert np.allclose(conferencia['probabilidade_churn'], esperado['probabilidade_churn'], atol=1e-8)
        assert conferencia['prediction'].tolist() == esperado['prediction'].tolist()
        display(conferencia.round(4))
        print('Conferência aprovada: o modelo recuperado reproduziu as previsões da entrega.')
    except Exception as exc:
        print(f'Não foi possível validar a recuperação do MLflow: {type(exc).__name__}: {exc}')
        print('É necessário verificar a configuração do MLflow; as previsões já geradas não foram alteradas.')

## 19. Síntese dos resultados

1. **Problema:** identificar clientes com maior risco de cancelamento, em um histórico com 26,7% de churn.
2. **Variáveis:** 24 candidatas → 19 após Information Value (corte 0,02) → **18** após remover redundância (correlação > 0,90). Nenhuma remoção custou desempenho.
3. **Modelo:** regressão logística com `C=1`, comparada com árvore de decisão, Random Forest, Gradient Boosting e XGBoost. O comparativo atualizado é apresentado na seção 12.3.
4. **Desempenho no teste:** AUC 0,84, Gini 0,69, KS 0,54. O top 10% de score cancela 2,8 vezes mais que a média.
5. **Calibração:** score médio de 26,3% e churn observado de 26,7% no teste; os resultados por faixa complementam a avaliação das probabilidades.
6. **Concentração:** no conjunto de teste, os 20% com maiores scores concentram aproximadamente metade dos cancelamentos observados.
7. **Corte:** `Yes` = risco acima da média da carteira (26,7%), próximo do ponto de maior separação (KS). Marca 79% dos cancelamentos, com 1 em cada 2 marcados cancelando de fato.
8. **Quem cancela:** pouco tempo de casa, contrato mensal, fibra, sem suporte técnico e sem segurança online.
9. **Inferência:** 1.409 clientes pontuados — 285 em risco alto, 298 médio e 826 baixo. PSI do score de 0,003, sem evidência relevante de mudança na distribuição.

**Limitações declaradas:** sem datas, a validação é aleatória e não prova com quanta antecedência o cancelamento é previsto; o modelo mostra associação, não causa — o efeito de uma oferta de retenção só se mede com teste A/B.

**Próximos passos:** datas de referência para validação temporal; histórico de troca de forma de pagamento; uso do `CustomerFeedback` para explicar o motivo do cancelamento (não para prever); teste A/B das ofertas por faixa de risco.

**Reprodutibilidade:** o pipeline aplica os mesmos tratamentos e o MLflow permite recuperar o modelo e o corte utilizados na inferência.


## 20. Registro de decisões

Todas as escolhas do projeto em um lugar. Para cada uma: o que foi decidido, a alternativa descartada e por quê.

| # | Etapa | Decisão | Alternativa descartada | Por quê |
|---|---|---|---|---|
| 1 | Leitura | Contar clientes por registro CSV (5.634), não por linha física (5.912) | Contar linhas | O texto do feedback quebra linhas dentro de um mesmo registro |
| 2 | Qualidade | Estimar `tenure` e `TotalCharges` um pelo outro; excluir 1 cliente sem os dois | Tratar nulo como zero | Nulo não é pagamento zero: o grupo sem total tinha tenure mediano de 25 meses |
| 3 | Vazamento | `CustomerFeedback` fora do modelo | Usar o texto como variável | Quando cita "cancel", 92,5% já cancelaram: o texto descreve o evento, não o antecipa |
| 4 | Divisão | 80% treino / 20% teste, estratificada, semente 42 | Usar a base inteira para treinar e avaliar | Sem prova independente, não há como saber se o modelo funciona em clientes novos |
| 5 | Validação | Validação cruzada com 5 partes, só no treino | 3 ou 10 partes | 5 oferece divisão estratificada e avaliação repetida com custo computacional moderado |
| 6 | Seleção | IV ≥ 0,02, calculado no treino (24 → 19) | Usar todas; calcular IV com o teste | Variáveis sem poder só somam ruído; o teste não pode influenciar escolhas |
| 7 | Redundância | Correlação > 0,90 remove a derivada (19 → 18) | Manter as duas | Variáveis duplicadas tornam os pesos da logística arbitrários |
| 8 | Preparação | One-hot com categoria de referência; padronização; tudo em pipeline | Ajustar a padronização na base inteira | Evita vazamento e torna os pesos comparáveis |
| 9 | Desbalanceamento | Não balancear; tratar no corte | SMOTE ou pesos de classe | Foi mantida a distribuição original e avaliado o corte; outras abordagens não foram priorizadas |
| 10 | Modelo | Regressão logística | Árvore, Random Forest e Gradient Boosting | Escolha orientada pela AUC na validação cruzada, estabilidade e interpretabilidade; comparar o XGBoost após a execução atualizada |
| 11 | Parâmetro | `C=1` | Outros valores da grade (0,01 a 10) | Nenhum superou o padrão além do ruído |
| 12 | Corte | Taxa média de churn (0,267), decidida na validação | 0,50; top 20% | Escolha documentada como referência inicial; o máximo KS é apresentado separadamente |
| 13 | Calibração | Verificada no teste | Supor que o score é probabilidade | Sustenta a leitura do corte e do score pelo negócio |
| 14 | Faixas | Alto (top 20%), médio (acima da média), baixo | Só Yes/No | As faixas de risco apoiam a priorização de clientes para retenção |
| 15 | Monitoramento | PSI do score e por variável, histórico × inferência, mesmo modelo | Não verificar | É o único teste possível sem saber quem cancelou na base nova |

| 16 | Pipeline e MLflow | Entrada bruta → ABT → seleção → preparo → regressão logística; modelo e corte registrados no MLflow | Manter código de tratamento separado da inferência | Mantém o mesmo tratamento na validação e na inferência. |


**Leitura:** o identificador da execução (`run_id`) permite recuperar o pipeline registrado. A conferência verifica se o modelo recuperado reproduz as probabilidades e classificações geradas na inferência.


## 21. Regras de negócio e aplicação dos resultados

As regras a seguir foram construídas a partir da análise exploratória do histórico, com **5.633 clientes após a exclusão do único registro sem `tenure` e `TotalCharges`**. Elas descrevem perfis observados; **não substituem a probabilidade calculada pela regressão logística**, não alteram o corte `Yes/No` e não demonstram que uma intervenção causará redução de cancelamentos.

### 21.1. Segmentos identificados no histórico

O **lift** compara a taxa de cancelamento do segmento com a média do histórico (aproximadamente **26,7%**). Por exemplo, lift 2 significa uma taxa de cancelamento duas vezes maior que a média. **Os segmentos se sobrepõem**: um mesmo cliente pode aparecer em várias regras, portanto as quantidades e as participações não devem ser somadas.

| Regra | Perfil do cliente | Clientes | Churn observado | Lift | % de todos os cancelamentos |
|---|---|---:|---:|---:|---:|
| R1 | Contrato mensal + fibra óptica + cheque eletrônico | 1.038 | 59,9% | 2,25 | 41,4% |
| R2 | Mensal + fibra + sem suporte técnico + sem segurança online | 1.221 | 60,8% | 2,28 | 49,4% |
| R3 | Mensal + cheque eletrônico + sem suporte técnico | 1.193 | 58,4% | 2,19 | 46,4% |
| R4 | Fibra + sem suporte técnico + sem segurança online | 1.423 | 55,1% | 2,06 | 52,2% |
| R5 | Mensal + fibra, independentemente do pagamento | 1.698 | 54,7% | 2,05 | 61,7% |
| R6 | Mensal + até 12 meses de relacionamento | 1.593 | 50,8% | 1,90 | 53,8% |
| R7 | Mensal + até 12 meses + fibra | 730 | 69,6% | 2,61 | 33,8% |
| R8 | Mensal + até 12 meses + cheque eletrônico | 754 | 62,2% | 2,33 | 31,2% |
| R9 | Mensal + até 12 meses + fibra + cheque eletrônico | 500 | 69,6% | 2,61 | 23,2% |
| R10 | Até 12 meses + mensalidade alta + fibra | 309 | 75,1% | 2,81 | 15,4% |
| R11 | Mensal + fibra + pagamento automático | 496 | 44,8% | 1,68 | 14,8% |
| R12 | Mensal + fibra + com suporte + com segurança | 64 | 29,7% | 1,11 | 1,3% |
| R13 | Mensal + DSL | 982 | 32,5% | 1,22 | 21,2% |
| R14 | Mensal + sem internet | 419 | 17,9% | 0,67 | 5,0% |
| R15 | Anual ou bienal + pagamento automático | 1.538 | 5,8% | 0,22 | 5,9% |
| R16 | Anual ou bienal, independentemente do pagamento | 2.534 | 7,1% | 0,27 | 12,0% |
| R17 | Cliente idoso + fibra + contrato mensal | 520 | 58,5% | 2,19 | 20,2% |

**Destaques:** o grupo R10 apresenta a maior taxa observada de cancelamento (75,1%). O grupo R15 apresenta uma das menores taxas (5,8%). As regras R1 e R2, consideradas em conjunto e sem duplicar clientes, abrangem cerca de 26% do histórico e concentram aproximadamente 56,5% dos cancelamentos. Na regra R10, a definição operacional de *mensalidade alta* precisa ser documentada antes de reutilizar a segmentação em outro conjunto de dados.

### 21.2. Estratégias de retenção propostas

As ações são **hipóteses de intervenção** para avaliação do negócio. A priorização deve considerar o score do modelo, a necessidade identificada, a viabilidade para o cliente e o custo da iniciativa.

| Perfil ou situação | Ação sugerida | Condição de aplicação |
|---|---|---|
| Alto risco + pagamento não automático | Oferecer migração para débito ou cartão automático | Verificar elegibilidade e aceitação do cliente; associação histórica não comprova redução causal do churn |
| Alto risco + contrato mensal | Avaliar oferta de migração para plano anual | Confirmar condições comerciais, benefício ao cliente e eventual desconto |
| Fibra + reclamação de preço ou oferta concorrente | Revisar plano e considerar proposta personalizada | Confirmar motivo do contato e demonstrar viabilidade financeira antes de conceder desconto |
| Sem suporte técnico ou segurança online | Avaliar período de experimentação de suporte ou proteção | Definir custos, duração e elegibilidade; mensurar resultado da oferta |
| Até 12 meses de relacionamento | Priorizar onboarding e acompanhamento | Testar se o acompanhamento reduz o cancelamento em comparação com grupo de controle |
| Risco intermediário | Considerar contato de menor custo, como e-mail ou SMS | Respeitar consentimento, capacidade operacional e política de comunicação |
| Contrato anual/bienal + pagamento automático, baixo score | Não priorizar contato ativo de retenção | Manter atendimento normal e monitoramento do comportamento |

**Critério financeiro:** não há custo padronizado de contato, desconto ou serviço, nem orçamento de retenção fornecido no desafio. Assim, **não foi calculado um desconto máximo comprovadamente rentável**. Uma proposta comercial deve explicitar o custo da ação, a receita ou margem que poderia ser preservada e, principalmente, a probabilidade **incremental** de evitar o cancelamento. A probabilidade de churn, isoladamente, não mede o efeito de um desconto. A efetividade das ofertas pode ser avaliada em teste controlado.

**Contratos:** o tempo de relacionamento é contado desde o primeiro contrato; contratos anuais e bienais são renovados automaticamente. O cliente pode cancelar antes do fim do período, **sem multa**. Por isso, os meses restantes do ciclo atual não equivalem a uma receita contratualmente garantida.

**Feedback:** os relatos textuais podem ajudar a investigar motivos de insatisfação e orientar atendimento, mas não integram o modelo preditivo principal. Não há data confiável de coleta em relação ao cancelamento, e a pesquisa não é recorrente, o que cria risco de vazamento temporal se o texto for usado como preditora.

*Fonte das regras: consolidação das análises e propostas do Grupo 6. As taxas são descritivas e as ações são recomendações sujeitas à validação operacional e financeira.*
